# Notebook 13 — Prize-Money Semantics and Availability

## Purpose

This notebook investigates the raw source prize-money field before it is used in the future database or in analysis.

The investigation begins without assuming that the field represents:

- the total race purse;
- the winner’s prize;
- the first-place share;
- an advertised or guaranteed fund;
- the amount actually paid; or
- a consistent concept across jurisdictions and periods.

The notebook will distinguish the raw source value from candidate and confirmed interpretations, measure coverage at runner-row and provisional-race level, test within-race consistency, document formatting and failure modes, and define a defensible canonicalisation and unresolved-case policy.

Provisional races will be identified using:

`date + course + off`

The supplied `race_id` will be retained as source lineage but will not be treated as a unique race key.

All raw-source queries must:

- use read-only SQLite access;
- query the `data` table; and
- exclude the header-like first row with `rowid <> 1`.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

DB_PATH = Path("../data/raw/form_2015-present/form_2015-present/raceform.db")

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    schema = pd.read_sql_query("PRAGMA table_info(data)", conn)

schema

,cid,name,type,notnull,dflt_value,pk
0,0,date,NUMERIC,0,None,0
1,1,course,TEXT,0,None,0
2,2,race_id,INTEGER,0,None,0
3,3,off,TEXT,0,None,0
4,4,race_name,TEXT,0,None,0
5,5,type,TEXT,0,None,0
6,6,class,TEXT,0,None,0
7,7,pattern,TEXT,0,None,0
8,8,rating_band,TEXT,0,None,0
9,9,age_band,TEXT,0,None,0


In [2]:
DATA_ROW_PREDICATE = "rowid <> 1"

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_row_profile = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS runner_rows,
            COUNT(prize) AS nonnull_prize_rows,
            SUM(CASE WHEN prize IS NULL THEN 1 ELSE 0 END) AS null_prize_rows,
            SUM(CASE WHEN prize = 0 THEN 1 ELSE 0 END) AS zero_prize_rows,
            COUNT(DISTINCT prize) AS distinct_nonnull_values,
            MIN(prize) AS minimum_prize,
            MAX(prize) AS maximum_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        """,
        conn,
    )

prize_row_profile

,runner_rows,nonnull_prize_rows,null_prize_rows,zero_prize_rows,distinct_nonnull_values,minimum_prize,maximum_prize
0,1851285,1851285,0,0,47215,0.12,€9975


In [3]:
# SQLite column declarations do not enforce a single storage type.
# Although `prize` is declared as INTEGER, the previous output showed both
# numeric values and text such as "€9975".
#
# This query groups the field by SQLite's actual per-value storage class so
# we can measure how much of the column is stored as integer, real, text,
# null, or another type before attempting any monetary parsing.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_storage_profile = pd.read_sql_query(
        f"""
        SELECT
            typeof(prize) AS sqlite_storage_class,
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT prize) AS distinct_values
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY typeof(prize)
        ORDER BY runner_rows DESC
        """,
        conn,
    )

prize_storage_profile

,sqlite_storage_class,runner_rows,distinct_values
0,text,1008181,1625
1,real,618026,41316
2,integer,225078,4274


In [4]:
# Inspect representative raw values from each SQLite storage class.
#
# We take the smallest and largest values according to SQLite's ordering,
# plus a sample of distinct values, to see what formats are present in the
# text, real, and integer populations.
#
# This is descriptive only: no monetary interpretation or currency assignment
# is attempted yet.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_storage_examples = pd.read_sql_query(
        f"""
        WITH distinct_prizes AS (
            SELECT DISTINCT
                typeof(prize) AS sqlite_storage_class,
                prize
            FROM data
            WHERE {DATA_ROW_PREDICATE}
        ),
        ranked AS (
            SELECT
                sqlite_storage_class,
                prize,
                ROW_NUMBER() OVER (
                    PARTITION BY sqlite_storage_class
                    ORDER BY prize
                ) AS ascending_rank,
                ROW_NUMBER() OVER (
                    PARTITION BY sqlite_storage_class
                    ORDER BY prize DESC
                ) AS descending_rank
            FROM distinct_prizes
        )
        SELECT
            sqlite_storage_class,
            prize
        FROM ranked
        WHERE ascending_rank <= 10
           OR descending_rank <= 10
        ORDER BY sqlite_storage_class, prize
        """,
        conn,
    )

prize_storage_examples

,sqlite_storage_class,prize
0,integer,6
1,integer,37
2,integer,42
3,integer,50
4,integer,56
5,integer,57
6,integer,58
7,integer,59
8,integer,63
9,integer,68


In [5]:
# Measure effective row-level availability rather than relying on NULL status.
#
# A prize value is treated as blank when its text representation becomes empty
# after trimming whitespace. Populated values are then split by SQLite storage
# class so we can see how much apparent coverage comes from text, real and
# integer values.
#
# This does not yet assume that any populated value is analytically usable.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_effective_coverage = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS runner_rows,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) = '' THEN 1
                    ELSE 0
                END
            ) AS blank_prize_rows,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> '' THEN 1
                    ELSE 0
                END
            ) AS nonblank_prize_rows,
            SUM(
                CASE
                    WHEN typeof(prize) = 'text'
                     AND TRIM(CAST(prize AS TEXT)) <> ''
                    THEN 1 ELSE 0
                END
            ) AS nonblank_text_rows,
            SUM(CASE WHEN typeof(prize) = 'real' THEN 1 ELSE 0 END)
                AS real_rows,
            SUM(CASE WHEN typeof(prize) = 'integer' THEN 1 ELSE 0 END)
                AS integer_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        """,
        conn,
    )

prize_effective_coverage

,runner_rows,blank_prize_rows,nonblank_prize_rows,nonblank_text_rows,real_rows,integer_rows
0,1851285,839715,1011570,168466,618026,225078


In [6]:
# Measure prize availability at provisional-race level.
#
# A provisional race is identified by date + course + off.
# For each race, we count runner rows, blank prize rows and nonblank prize rows.
#
# This distinguishes:
# - races with prize populated on every runner row;
# - races with prize blank on every runner row; and
# - races with mixed blank and populated runner rows.
#
# Mixed availability would indicate that prize cannot safely be treated as a
# simple race-level field without further investigation.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_race_coverage = pd.read_sql_query(
        f"""
        WITH race_prize_profile AS (
            SELECT
                date,
                course,
                off,
                COUNT(*) AS runner_rows,
                SUM(
                    CASE
                        WHEN TRIM(CAST(prize AS TEXT)) = '' THEN 1
                        ELSE 0
                    END
                ) AS blank_prize_rows,
                SUM(
                    CASE
                        WHEN TRIM(CAST(prize AS TEXT)) <> '' THEN 1
                        ELSE 0
                    END
                ) AS nonblank_prize_rows
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY date, course, off
        )
        SELECT
            CASE
                WHEN nonblank_prize_rows = runner_rows
                    THEN 'all_runner_rows_nonblank'
                WHEN blank_prize_rows = runner_rows
                    THEN 'all_runner_rows_blank'
                ELSE 'mixed_blank_and_nonblank'
            END AS availability_pattern,
            COUNT(*) AS provisional_races,
            SUM(runner_rows) AS runner_rows,
            SUM(nonblank_prize_rows) AS nonblank_prize_rows,
            SUM(blank_prize_rows) AS blank_prize_rows
        FROM race_prize_profile
        GROUP BY availability_pattern
        ORDER BY provisional_races DESC
        """,
        conn,
    )

prize_race_coverage

,availability_pattern,provisional_races,runner_rows,nonblank_prize_rows,blank_prize_rows
0,mixed_blank_and_nonblank,161398,1686647,846995,839652
1,all_runner_rows_nonblank,27633,164575,164575,0
2,all_runner_rows_blank,12,63,0,63


In [7]:
# Profile the populated prize values within each provisional race.
#
# If `prize` were normally a race-level purse, we would expect either:
# - one repeated nonblank value across all runner rows; or
# - one populated row and blanks elsewhere because of a source-layout choice.
#
# If it represents money awarded to individual finishers, we would instead
# expect several populated runner rows within a race, often with different
# amounts.
#
# This cell therefore counts:
# - the number of nonblank prize rows per race; and
# - the number of distinct nonblank raw prize values per race.
#
# No attempt is yet made to parse currencies or compare monetary amounts.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_within_race_profile = pd.read_sql_query(
        f"""
        WITH race_prize_profile AS (
            SELECT
                date,
                course,
                off,
                COUNT(*) AS runner_rows,
                SUM(
                    CASE
                        WHEN TRIM(CAST(prize AS TEXT)) <> '' THEN 1
                        ELSE 0
                    END
                ) AS nonblank_prize_rows,
                COUNT(
                    DISTINCT CASE
                        WHEN TRIM(CAST(prize AS TEXT)) <> ''
                        THEN CAST(prize AS TEXT)
                    END
                ) AS distinct_nonblank_prize_values
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY date, course, off
        )
        SELECT
            nonblank_prize_rows,
            distinct_nonblank_prize_values,
            COUNT(*) AS provisional_races,
            SUM(runner_rows) AS runner_rows
        FROM race_prize_profile
        GROUP BY
            nonblank_prize_rows,
            distinct_nonblank_prize_values
        ORDER BY
            provisional_races DESC,
            nonblank_prize_rows,
            distinct_nonblank_prize_values
        LIMIT 30
        """,
        conn,
    )

prize_within_race_profile

,nonblank_prize_rows,distinct_nonblank_prize_values,provisional_races,runner_rows
0,5,5,57030,571077
1,4,4,53680,459599
2,6,6,34128,388575
3,8,4,7558,84050
4,7,7,6256,76660
5,6,5,5985,53931
6,8,5,4577,48266
7,3,3,4344,21111
8,8,6,2362,24875
9,7,5,1407,10757


In [8]:
# Inspect representative races with common prize-population patterns.
#
# Including finishing position lets us test the candidate interpretation that
# `prize` records the amount awarded to each runner, rather than a race-level
# purse or advertised fund.
#
# We select a small number of races with:
# - five populated rows and five distinct values;
# - four populated rows and four distinct values; and
# - eight populated rows but only four distinct values.
#
# Raw prize values are preserved exactly as stored.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_race_examples = pd.read_sql_query(
        f"""
        WITH race_profiles AS (
            SELECT
                date,
                course,
                off,
                SUM(
                    CASE
                        WHEN TRIM(CAST(prize AS TEXT)) <> '' THEN 1
                        ELSE 0
                    END
                ) AS nonblank_prize_rows,
                COUNT(
                    DISTINCT CASE
                        WHEN TRIM(CAST(prize AS TEXT)) <> ''
                        THEN CAST(prize AS TEXT)
                    END
                ) AS distinct_nonblank_prize_values
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY date, course, off
        ),
        selected_races AS (
            SELECT
                date,
                course,
                off,
                nonblank_prize_rows,
                distinct_nonblank_prize_values,
                ROW_NUMBER() OVER (
                    PARTITION BY
                        nonblank_prize_rows,
                        distinct_nonblank_prize_values
                    ORDER BY date, course, off
                ) AS example_number
            FROM race_profiles
            WHERE
                (nonblank_prize_rows = 5 AND distinct_nonblank_prize_values = 5)
                OR
                (nonblank_prize_rows = 4 AND distinct_nonblank_prize_values = 4)
                OR
                (nonblank_prize_rows = 8 AND distinct_nonblank_prize_values = 4)
        )
        SELECT
            d.date,
            d.course,
            d.off,
            d.race_name,
            d.pos,
            d.horse,
            d.prize,
            typeof(d.prize) AS prize_storage_class
        FROM data AS d
        JOIN selected_races AS s
          ON d.date = s.date
         AND d.course = s.course
         AND d.off = s.off
        WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
          AND s.example_number = 1
        ORDER BY
            s.nonblank_prize_rows,
            s.distinct_nonblank_prize_values,
            d.date,
            d.course,
            d.off,
            CASE
                WHEN typeof(d.pos) IN ('integer', 'real') THEN d.pos
                ELSE 999
            END,
            d.rowid
        """,
        conn,
    )

prize_race_examples

,date,course,off,race_name,pos,horse,prize,prize_storage_class
0,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,1,Definitly Red (IRE),4873.5,real
1,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,2,LAigle Royal (GER),1431,integer
2,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,3,Celtic Agent (GB),715.5,real
3,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,4,Aristo Du Plessis (FR),357.75,real
4,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,5,Knight Bachelor (GB),,text
5,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,6,Palm Grey (IRE),,text
6,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,7,Lowcarr Motion (GB),,text
7,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,8,Tomorrows Legend (GB),,text
8,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,9,Masirann (IRE),,text
9,2015-01-01,Catterick,12:30,Happy New Year Novices Hurdle,PU,Tickenwolf (IRE),,text


In [9]:
# Test how prize availability relates to recorded finishing position.
#
# The examples suggest that `prize` is populated for runners receiving an
# individual placing payment. Across the full source, we therefore classify
# positions into:
# - positive numeric finishes;
# - zero;
# - disqualification;
# - other textual outcomes such as PU, F or UR; and
# - blank positions.
#
# For each category, we count blank and nonblank prize rows. This tests the
# runner-prize interpretation and exposes exceptions requiring investigation.
#
# Prize remains unparsed and currency-neutral at this stage.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_by_position_type = pd.read_sql_query(
        f"""
        SELECT
            CASE
                WHEN typeof(pos) IN ('integer', 'real') AND pos > 0
                    THEN 'positive_numeric_finish'
                WHEN typeof(pos) IN ('integer', 'real') AND pos = 0
                    THEN 'zero_position'
                WHEN UPPER(TRIM(CAST(pos AS TEXT))) IN ('DSQ', 'DQ')
                    THEN 'disqualified'
                WHEN TRIM(CAST(pos AS TEXT)) = ''
                    THEN 'blank_position'
                ELSE 'other_text_outcome'
            END AS position_category,
            COUNT(*) AS runner_rows,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> '' THEN 1
                    ELSE 0
                END
            ) AS nonblank_prize_rows,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) = '' THEN 1
                    ELSE 0
                END
            ) AS blank_prize_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY position_category
        ORDER BY runner_rows DESC
        """,
        conn,
    )

prize_by_position_type

,position_category,runner_rows,nonblank_prize_rows,blank_prize_rows
0,positive_numeric_finish,1756666,1011430,745236
1,other_text_outcome,93992,140,93852
2,disqualified,619,0,619
3,zero_position,8,0,8


In [10]:
# Inspect every distinct textual finishing-position code that carries a prize.
#
# The previous profile found 140 populated prize rows outside positive numeric
# finishing positions. Before treating them as semantic exceptions, we need to
# see whether the `pos` field contains alternative placed-finish encodings,
# dead-heat markers, source errors, or genuine non-completion outcomes.
#
# This cell groups those rows by the raw position value and shows their
# frequency, prize range and date range.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    prize_text_position_exceptions = pd.read_sql_query(
        f"""
        SELECT
            CAST(pos AS TEXT) AS raw_pos,
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT CAST(prize AS TEXT)) AS distinct_prize_values,
            MIN(prize) AS minimum_raw_prize,
            MAX(prize) AS maximum_raw_prize,
            MIN(date) AS first_date,
            MAX(date) AS last_date
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND TRIM(CAST(prize AS TEXT)) <> ''
          AND NOT (
              typeof(pos) IN ('integer', 'real')
              AND pos > 0
          )
          AND UPPER(TRIM(CAST(pos AS TEXT))) NOT IN ('DSQ', 'DQ')
        GROUP BY CAST(pos AS TEXT)
        ORDER BY runner_rows DESC, raw_pos
        """,
        conn,
    )

prize_text_position_exceptions

,raw_pos,runner_rows,distinct_prize_values,minimum_raw_prize,maximum_raw_prize,first_date,last_date
0,PU,82,71,66.93,€5700,2015-01-03,2026-05-16
1,F,43,39,361.27,€6000,2015-04-19,2026-05-16
2,UR,11,11,247.83,9124.09,2016-04-30,2026-01-06
3,BD,3,3,361.27,6000,2018-10-21,2025-05-10
4,RR,1,1,1851.85,1851.85,2018-04-14,2018-04-14


In [11]:
# Inspect representative races where a non-finisher carries a nonblank prize.
#
# We include every runner from a small sample of affected races so that the
# anomalous row can be compared with the race's normal finishing positions,
# prize schedule and runner ordering.
#
# The earliest race for each anomalous outcome code is selected. This should
# help distinguish genuine prize payments from row misalignment or source error.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    anomalous_prize_race_examples = pd.read_sql_query(
        f"""
        WITH anomalous_races AS (
            SELECT
                date,
                course,
                off,
                CAST(pos AS TEXT) AS anomalous_pos,
                ROW_NUMBER() OVER (
                    PARTITION BY CAST(pos AS TEXT)
                    ORDER BY date, course, off
                ) AS example_number
            FROM data
            WHERE {DATA_ROW_PREDICATE}
              AND TRIM(CAST(prize AS TEXT)) <> ''
              AND CAST(pos AS TEXT) IN ('PU', 'F', 'UR', 'BD', 'RR')
        )
        SELECT
            d.date,
            d.course,
            d.off,
            d.race_name,
            d.ran,
            d.pos,
            d.horse,
            d.prize,
            typeof(d.prize) AS prize_storage_class
        FROM data AS d
        JOIN anomalous_races AS a
          ON d.date = a.date
         AND d.course = a.course
         AND d.off = a.off
        WHERE d.rowid <> 1
          AND a.example_number = 1
        ORDER BY
            a.anomalous_pos,
            d.date,
            d.course,
            d.off,
            d.rowid
        """,
        conn,
    )

anomalous_prize_race_examples

,date,course,off,race_name,ran,pos,horse,prize,prize_storage_class
0,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,1,Patricia Anne (NZ),7947.98,real
1,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,2,The Regiment (AUS),2456.65,real
2,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,3,Bouffon (AUS),1156.07,real
3,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,4,Mr Monaco (AUS),650.29,real
4,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,5,Brutus Rex (AUS),433.53,real
5,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,6,Bettys Thrills (AUS),361.27,real
6,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,7,The Queens Regent (AUS),361.27,real
7,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,8,Fuse (AUS),361.27,real
8,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,BD,Vin Fiz (AUS),361.27,real
9,2018-10-21,Cranbourne (AUS),4:25,Highview Accounting And Financial BM64 Handica...,10,F,Frankly Harvey (AUS),361.27,real


In [12]:
# Add the repository's `src` directory to the notebook import path.
#
# The notebook is stored under `notebooks/`, so its parent directory is the
# repository root and `../src` contains the `inside_rails` package.
#
# This changes only the current Python session. It does not modify the project
# or install anything into the environment.

import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Repository root: {REPO_ROOT}")
print(f"Source path added: {SRC_PATH}")
print(f"Source path exists: {SRC_PATH.exists()}")

Repository root: /home/rob/Documents/inside-rails-horse-racing
Source path added: /home/rob/Documents/inside-rails-horse-racing/src
Source path exists: True


In [13]:
# Inspect the reusable jurisdiction functions before using them.
#
# We need their exact signatures and documentation so Notebook 13 can reuse
# the established project logic rather than recreating jurisdiction rules.

import inspect

from inside_rails.course_jurisdiction import (
    derive_candidate_course_label,
    derive_candidate_race_jurisdiction,
)

print(inspect.signature(derive_candidate_course_label))
print(inspect.getdoc(derive_candidate_course_label))
print()
print(inspect.signature(derive_candidate_race_jurisdiction))
print(inspect.getdoc(derive_candidate_race_jurisdiction))

(course_name)
Remove only a recognised terminal jurisdiction suffix.

(row)
Derive candidate jurisdiction and retain the supporting rule.


In [14]:
# Inspect the reusable jurisdiction implementation.
#
# This avoids guessing which fields must be present in the row supplied to
# `derive_candidate_race_jurisdiction` and lets us preserve its returned
# evidence fields correctly in the prize-money audit.

print(inspect.getsource(derive_candidate_race_jurisdiction))

def derive_candidate_race_jurisdiction(row):
    """Derive candidate jurisdiction and retain the supporting rule."""
    course_name = str(row["course"])
    race_date = str(row["date"])
    race_type = str(row["type"])
    race_name = str(row["race_name"])

    terminal_code = extract_terminal_jurisdiction_code(course_name)

    if terminal_code is not None:
        return pd.Series(
            [
                terminal_code_to_jurisdiction[terminal_code],
                "explicit_terminal_course_code",
            ]
        )

    if course_name in historical_course_to_code:
        historical_code = historical_course_to_code[course_name]

        return pd.Series(
            [
                terminal_code_to_jurisdiction[historical_code],
                "historical_suffixed_course_link",
            ]
        )

    if course_name in curated_british_course_configurations:
        return pd.Series(
            [
                "Great Britain",
                "curated_british_

In [15]:
# Derive jurisdiction for the 140 non-finisher rows carrying prize money.
#
# These rows are rare but potentially meaningful. Grouping them by candidate
# jurisdiction and outcome code will show whether they are concentrated in
# jurisdictions where all starters may receive a minimum payment.
#
# We preserve the jurisdiction rule because the result is candidate evidence,
# not an unquestioned source attribute.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    anomalous_prize_rows = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            race_name,
            type,
            CAST(pos AS TEXT) AS raw_pos,
            prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND TRIM(CAST(prize AS TEXT)) <> ''
          AND CAST(pos AS TEXT) IN ('PU', 'F', 'UR', 'BD', 'RR')
        """,
        conn,
    )

anomalous_prize_rows[
    ["candidate_jurisdiction", "jurisdiction_rule"]
] = anomalous_prize_rows.apply(
    derive_candidate_race_jurisdiction,
    axis=1,
)

anomalous_prize_jurisdiction_profile = (
    anomalous_prize_rows
    .groupby(
        ["candidate_jurisdiction", "raw_pos"],
        dropna=False,
    )
    .agg(
        runner_rows=("prize", "size"),
        provisional_races=("off", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
    .sort_values(
        ["runner_rows", "candidate_jurisdiction", "raw_pos"],
        ascending=[False, True, True],
    )
)

anomalous_prize_jurisdiction_profile

,candidate_jurisdiction,raw_pos,runner_rows,provisional_races,first_date,last_date
17,United States,PU,48,48,2015-01-03,2026-04-04
5,France,F,31,31,2015-04-19,2026-05-16
6,France,PU,14,14,2017-11-04,2026-05-16
9,Great Britain,PU,11,11,2015-09-25,2024-12-07
11,Ireland,F,7,7,2021-01-01,2026-04-30
19,United States,UR,6,6,2017-07-30,2022-11-04
2,Australia,PU,3,3,2022-01-29,2026-03-07
7,France,UR,3,3,2018-07-15,2026-01-06
16,United States,F,3,3,2017-03-04,2026-02-06
4,Canada,PU,2,2,2019-06-22,2021-12-05


In [16]:
# Measure prize availability by candidate jurisdiction at provisional-race level.
#
# Jurisdiction is derived once per provisional race rather than once per runner
# row. The race-level table then records:
# - total runner rows;
# - populated prize rows;
# - whether the race has any populated prize;
# - whether every runner row has a populated prize; and
# - how many distinct populated raw prize values occur.
#
# This establishes whether prize availability and structure vary materially by
# jurisdiction before we attempt currency assignment or monetary parsing.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    race_prize_profiles = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            MIN(race_name) AS race_name,
            MIN(type) AS type,
            COUNT(*) AS runner_rows,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> '' THEN 1
                    ELSE 0
                END
            ) AS nonblank_prize_rows,
            COUNT(
                DISTINCT CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> ''
                    THEN CAST(prize AS TEXT)
                END
            ) AS distinct_nonblank_prize_values
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY date, course, off
        """,
        conn,
    )

race_prize_profiles[
    ["candidate_jurisdiction", "jurisdiction_rule"]
] = race_prize_profiles.apply(
    derive_candidate_race_jurisdiction,
    axis=1,
)

prize_coverage_by_jurisdiction = (
    race_prize_profiles
    .groupby("candidate_jurisdiction", dropna=False)
    .agg(
        provisional_races=("off", "size"),
        runner_rows=("runner_rows", "sum"),
        nonblank_prize_rows=("nonblank_prize_rows", "sum"),
        races_with_any_prize=(
            "nonblank_prize_rows",
            lambda values: (values > 0).sum(),
        ),
        races_with_all_rows_prized=(
            "nonblank_prize_rows",
            lambda values: (
                values.to_numpy()
                == race_prize_profiles.loc[values.index, "runner_rows"].to_numpy()
            ).sum(),
        ),
    )
    .reset_index()
)

prize_coverage_by_jurisdiction["race_coverage_pct"] = (
    100
    * prize_coverage_by_jurisdiction["races_with_any_prize"]
    / prize_coverage_by_jurisdiction["provisional_races"]
).round(2)

prize_coverage_by_jurisdiction.sort_values(
    ["provisional_races", "candidate_jurisdiction"],
    ascending=[False, True],
).head(30)

,candidate_jurisdiction,provisional_races,runner_rows,nonblank_prize_rows,races_with_any_prize,races_with_all_rows_prized,race_coverage_pct
12,Great Britain,111634,984757,561852,111623,16402,99.99
16,Ireland,30783,356950,168466,30782,2509,100.00
10,France,20533,228332,115048,20533,1474,100.00
14,Hong Kong,7481,90975,39922,7481,32,100.00
34,United States,6105,50145,46675,6105,4992,100.00
1,Australia,4059,46156,34656,4059,1286,100.00
33,United Arab Emirates,2451,27890,13245,2451,106,100.00
18,Japan,1559,22551,7794,1559,33,100.00
11,Germany,731,6368,3522,731,56,100.00
5,Canada,471,3819,3765,471,445,100.00


In [17]:
# Inspect the 12 provisional races with no populated prize values.
#
# Race-level prize coverage is almost complete, so these races form a small,
# explicit unresolved set. We retain their descriptive fields and jurisdiction
# evidence to see whether the missingness is concentrated by date, venue,
# jurisdiction, race type, or another identifiable source pattern.

blank_prize_races = (
    race_prize_profiles.loc[
        race_prize_profiles["nonblank_prize_rows"] == 0,
        [
            "date",
            "course",
            "off",
            "race_name",
            "type",
            "runner_rows",
            "candidate_jurisdiction",
            "jurisdiction_rule",
        ],
    ]
    .sort_values(["date", "course", "off"])
    .reset_index(drop=True)
)

blank_prize_races

,date,course,off,race_name,type,runner_rows,candidate_jurisdiction,jurisdiction_rule
0,2015-09-24,Newmarket,5:55,Newmarket Challenge Whip (A Handicap),Flat,5,Great Britain,established_unsuffixed_british_course
1,2016-09-22,Newmarket,5:55,Newmarket Challenge Whip (A Handicap),Flat,3,Great Britain,established_unsuffixed_british_course
2,2017-09-28,Newmarket,6:05,Newmarket Challenge Whip Handicap,Flat,3,Great Britain,established_unsuffixed_british_course
3,2018-09-27,Newmarket,5:55,Newmarket Challenge Whip Handicap,Flat,5,Great Britain,established_unsuffixed_british_course
4,2019-09-26,Newmarket,5:50,Newmarket Challenge Whip Handicap,Flat,6,Great Britain,established_unsuffixed_british_course
5,2020-09-24,Newmarket,5:20,Newmarket Challenge Whip Handicap,Flat,5,Great Britain,established_unsuffixed_british_course
6,2021-09-23,Newmarket,5:05,Newmarket Challenge Whip Handicap,Flat,7,Great Britain,established_unsuffixed_british_course
7,2021-10-26,Curragh (IRE),4:40,Horseware Student Derby,Flat,12,Ireland,explicit_terminal_course_code
8,2022-09-22,Newmarket,5:20,Newmarket Challenge Whip Handicap,Flat,5,Great Britain,established_unsuffixed_british_course
9,2023-09-28,Newmarket,5:53,Newmarket Challenge Whip Handicap,Flat,5,Great Britain,established_unsuffixed_british_course


In [18]:
# Profile the raw formatting families used by populated text prize values.
#
# The `prize` column contains 168,466 nonblank text rows. This cell classifies
# them by visible currency marker and basic punctuation so we can establish:
# - which currency symbols or codes occur;
# - whether thousands separators are used;
# - whether decimal fractions occur; and
# - whether any text values contain unexpected non-monetary wording.
#
# The categories describe raw source formatting only. They do not yet assign
# currencies to numeric values or claim that the amounts are comparable.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    text_prize_format_profile = pd.read_sql_query(
        f"""
        SELECT
            CASE
                WHEN prize LIKE '€%' THEN 'euro_symbol'
                WHEN prize LIKE '£%' THEN 'pound_symbol'
                WHEN prize LIKE '$%' THEN 'dollar_symbol'
                WHEN prize GLOB '*[A-Za-z]*' THEN 'contains_letters'
                ELSE 'other_text_format'
            END AS text_format_family,
            SUM(CASE WHEN prize LIKE '%,%' THEN 1 ELSE 0 END)
                AS rows_with_comma,
            SUM(CASE WHEN prize LIKE '%.%' THEN 1 ELSE 0 END)
                AS rows_with_decimal_point,
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT prize) AS distinct_raw_values,
            MIN(LENGTH(prize)) AS minimum_length,
            MAX(LENGTH(prize)) AS maximum_length
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(prize) = 'text'
          AND TRIM(prize) <> ''
        GROUP BY text_format_family
        ORDER BY runner_rows DESC
        """,
        conn,
    )

text_prize_format_profile

,text_format_family,rows_with_comma,rows_with_decimal_point,runner_rows,distinct_raw_values,minimum_length,maximum_length
0,euro_symbol,2159,9573,168466,1624,3,9


In [19]:
# Test whether populated text prize values are confined to Ireland.
#
# The populated text-row count exactly matches Ireland's populated prize-row
# count, and every populated text value begins with the euro symbol.
#
# This cell derives candidate jurisdiction for races containing at least one
# text-stored prize value, then counts the affected races and runner rows.
# It will show whether text storage is an Irish formatting convention or
# whether euro strings also occur in other jurisdictions.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    text_prize_races = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            MIN(race_name) AS race_name,
            MIN(type) AS type,
            COUNT(*) AS runner_rows,
            SUM(
                CASE
                    WHEN typeof(prize) = 'text'
                     AND TRIM(CAST(prize AS TEXT)) <> ''
                    THEN 1 ELSE 0
                END
            ) AS populated_text_prize_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY date, course, off
        HAVING populated_text_prize_rows > 0
        """,
        conn,
    )

text_prize_races[
    ["candidate_jurisdiction", "jurisdiction_rule"]
] = text_prize_races.apply(
    derive_candidate_race_jurisdiction,
    axis=1,
)

text_prize_jurisdiction_profile = (
    text_prize_races
    .groupby("candidate_jurisdiction", dropna=False)
    .agg(
        provisional_races=("off", "size"),
        runner_rows=("runner_rows", "sum"),
        populated_text_prize_rows=("populated_text_prize_rows", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
    .sort_values(
        ["populated_text_prize_rows", "candidate_jurisdiction"],
        ascending=[False, True],
    )
)

text_prize_jurisdiction_profile

,candidate_jurisdiction,provisional_races,runner_rows,populated_text_prize_rows,first_date,last_date
0,Ireland,30782,356938,168466,2015-01-01,2026-05-27


In [20]:
# Profile prize storage classes by jurisdiction efficiently.
#
# Jurisdiction has already been derived once for each provisional race in
# `race_prize_profiles`. We therefore aggregate storage classes in SQLite at
# race level, then merge those counts with the existing race-level jurisdiction
# assignments.
#
# This avoids applying Python jurisdiction logic to more than one million
# runner rows.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    race_storage_counts = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> ''
                     AND typeof(prize) = 'text'
                    THEN 1 ELSE 0
                END
            ) AS text_prize_rows,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> ''
                     AND typeof(prize) = 'real'
                    THEN 1 ELSE 0
                END
            ) AS real_prize_rows,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> ''
                     AND typeof(prize) = 'integer'
                    THEN 1 ELSE 0
                END
            ) AS integer_prize_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY date, course, off
        """,
        conn,
    )

storage_with_jurisdiction = race_storage_counts.merge(
    race_prize_profiles[
        ["date", "course", "off", "candidate_jurisdiction"]
    ],
    on=["date", "course", "off"],
    how="left",
    validate="one_to_one",
)

prize_storage_by_jurisdiction = (
    storage_with_jurisdiction
    .groupby("candidate_jurisdiction", dropna=False)[
        ["text_prize_rows", "real_prize_rows", "integer_prize_rows"]
    ]
    .sum()
    .reset_index()
    .sort_values(
        ["text_prize_rows", "real_prize_rows", "integer_prize_rows"],
        ascending=False,
    )
)

prize_storage_by_jurisdiction

,candidate_jurisdiction,text_prize_rows,real_prize_rows,integer_prize_rows
16,Ireland,168466,0,0
12,Great Britain,0,348802,213050
10,France,0,111099,3949
34,United States,0,42106,4569
14,Hong Kong,0,39377,545
1,Australia,0,34476,180
33,United Arab Emirates,0,13094,151
18,Japan,0,7738,56
5,Canada,0,3670,95
11,Germany,0,3505,17


In [21]:
# Recreate the populated numeric prize dataset used by the precision analysis.
#
# Numeric prize values are stored by SQLite as INTEGER or REAL. Irish
# euro-prefixed text values are intentionally excluded from this analysis.
#
# The precision cell also groups by candidate jurisdiction, so we attach the
# governed race-level jurisdiction already calculated in race_prize_profiles
# using the provisional race key:
#
#     date + course + off

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    numeric_prize_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            typeof(prize) AS prize_storage_type,
            CAST(prize AS REAL) AS numeric_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(prize) IN ('integer', 'real')
          AND TRIM(CAST(prize AS TEXT)) <> ''
        """,
        conn,
    )

# Create one jurisdiction assignment per provisional race before merging.
numeric_prize_race_jurisdictions = (
    race_prize_profiles[
        [
            "date",
            "course",
            "off",
            "candidate_jurisdiction",
        ]
    ]
    .drop_duplicates()
)

numeric_prize_rows = numeric_prize_rows.merge(
    numeric_prize_race_jurisdictions,
    on=["date", "course", "off"],
    how="left",
    validate="many_to_one",
)

numeric_prize_rows[
    [
        "source_rowid",
        "date",
        "course",
        "off",
        "candidate_jurisdiction",
        "prize_storage_type",
        "numeric_prize",
    ]
].head()

,source_rowid,date,course,off,candidate_jurisdiction,prize_storage_type,numeric_prize
0,2,2015-01-01,Catterick,12:30,Great Britain,real,4873.50
1,43,2015-01-01,Laurel Park (USA),7:52,United States,real,14615.38
2,44,2015-01-01,Laurel Park (USA),7:52,United States,real,5384.62
3,45,2015-01-01,Laurel Park (USA),7:52,United States,real,2820.51
4,46,2015-01-01,Laurel Park (USA),7:52,United States,real,1538.46


In [22]:
# Classify the observed decimal precision of numeric prize values.
#
# We test whether each value is effectively unchanged when rounded to:
# - zero decimal places;
# - one decimal place; or
# - two decimal places.
#
# Values not matching any of those tests are retained as having more than
# two apparent decimal places. A small tolerance protects against ordinary
# floating-point representation noise.

import numpy as np

values = numeric_prize_rows["numeric_prize"].astype(float)

numeric_prize_rows["precision_family"] = np.select(
    [
        np.isclose(values, values.round(0), atol=1e-9),
        np.isclose(values, values.round(1), atol=1e-9),
        np.isclose(values, values.round(2), atol=1e-9),
    ],
    [
        "whole_number",
        "one_decimal_place",
        "two_decimal_places",
    ],
    default="more_than_two_decimals",
)

numeric_precision_profile = (
    numeric_prize_rows
    .groupby(
        ["candidate_jurisdiction", "precision_family"],
        dropna=False,
    )
    .agg(
        runner_rows=("numeric_prize", "size"),
        distinct_values=("numeric_prize", "nunique"),
        minimum_value=("numeric_prize", "min"),
        maximum_value=("numeric_prize", "max"),
    )
    .reset_index()
    .sort_values(
        ["candidate_jurisdiction", "runner_rows"],
        ascending=[True, False],
    )
)

numeric_precision_profile

,candidate_jurisdiction,precision_family,runner_rows,distinct_values,minimum_value,maximum_value
0,Argentina,one_decimal_place,1449,607,62.50,43508.53
1,Argentina,two_decimal_places,703,299,217.54,4846.05
2,Argentina,whole_number,607,285,125.00,255232.26
3,Australia,one_decimal_place,15193,918,219.90,47192.51
5,Australia,whole_number,11129,2156,500.00,4264971.75
...,...,...,...,...,...,...
94,United States,one_decimal_place,16648,2221,56.30,49593.50
95,United States,two_decimal_places,13031,1305,0.29,4984.25
97,Uruguay,one_decimal_place,183,138,1081.81,39080.52
99,Uruguay,whole_number,74,61,3386.99,222222.22


In [23]:
# Recalculate decimal precision using absolute tolerance only.
#
# `np.isclose()` normally also applies a relative tolerance. For large monetary
# values, that can incorrectly classify a value such as 43508.53 as matching
# 43508.5 or even a whole number.
#
# Setting `rtol=0` ensures that classification depends only on the very small
# absolute tolerance used to absorb floating-point representation noise.

values = numeric_prize_rows["numeric_prize"].astype(float)

numeric_prize_rows["precision_family"] = np.select(
    [
        np.isclose(values, values.round(0), rtol=0, atol=1e-9),
        np.isclose(values, values.round(1), rtol=0, atol=1e-9),
        np.isclose(values, values.round(2), rtol=0, atol=1e-9),
    ],
    [
        "whole_number",
        "one_decimal_place",
        "two_decimal_places",
    ],
    default="more_than_two_decimals",
)

numeric_precision_profile = (
    numeric_prize_rows
    .groupby(
        ["candidate_jurisdiction", "precision_family"],
        dropna=False,
    )
    .agg(
        runner_rows=("numeric_prize", "size"),
        distinct_values=("numeric_prize", "nunique"),
        minimum_value=("numeric_prize", "min"),
        maximum_value=("numeric_prize", "max"),
    )
    .reset_index()
    .sort_values(
        ["candidate_jurisdiction", "runner_rows"],
        ascending=[True, False],
    )
)

numeric_precision_profile

,candidate_jurisdiction,precision_family,runner_rows,distinct_values,minimum_value,maximum_value
1,Argentina,two_decimal_places,2297,1020,217.54,255232.26
0,Argentina,one_decimal_place,289,106,62.50,93725.10
2,Argentina,whole_number,173,65,125.00,156250.00
4,Australia,two_decimal_places,29888,3055,141.36,4264971.75
3,Australia,one_decimal_place,4588,299,219.90,2599009.90
...,...,...,...,...,...,...
87,United States,whole_number,4569,842,125.00,2912000.00
85,United States,one_decimal_place,3660,565,56.30,3149606.30
89,Uruguay,two_decimal_places,259,201,63.25,222222.22
88,Uruguay,one_decimal_place,28,21,1904.10,64767.70


In [24]:
# Summarise decimal precision across all numeric prize rows.
#
# This confirms whether any populated numeric values require more than two
# decimal places and measures how common whole-number, one-decimal and
# two-decimal amounts are.
#
# A maximum of two decimals would support safe storage in decimal monetary
# units, but it would not by itself identify the currency or prove whether
# foreign amounts have already been converted.

numeric_precision_summary = (
    numeric_prize_rows
    .groupby("precision_family", dropna=False)
    .agg(
        runner_rows=("numeric_prize", "size"),
        distinct_values=("numeric_prize", "nunique"),
        jurisdictions=("candidate_jurisdiction", "nunique"),
        minimum_value=("numeric_prize", "min"),
        maximum_value=("numeric_prize", "max"),
    )
    .reset_index()
    .sort_values("runner_rows", ascending=False)
)

numeric_precision_summary

,precision_family,runner_rows,distinct_values,jurisdictions,minimum_value,maximum_value
1,two_decimal_places,428761,34600,33,0.12,7874015.75
2,whole_number,225078,4274,26,6.00,8000000.00
0,one_decimal_place,189265,6716,32,0.90,3149606.30


In [25]:
# Test whether prize storage and decimal formatting change over time.
#
# We focus on the five largest jurisdictions:
# Great Britain, Ireland, France, the United States and Australia.
#
# For each year and jurisdiction, this cell counts populated prize rows by
# SQLite storage class. This will show whether:
# - Ireland's euro-text convention is stable throughout the source period;
# - numeric jurisdictions switch formats in particular years; or
# - integer/real differences simply coexist because whole amounts are stored
#   as INTEGER while fractional amounts are stored as REAL.
#
# This is still a raw-format audit. It does not assign currency to numeric
# values or assume that foreign amounts are local-currency figures.

numeric_prize_years = numeric_prize_rows.copy()
numeric_prize_years["year"] = (
    numeric_prize_years["date"].astype(str).str[:4]
)

numeric_storage_by_year = (
    storage_with_jurisdiction
    .assign(year=lambda frame: frame["date"].astype(str).str[:4])
    .loc[
        lambda frame: frame["candidate_jurisdiction"].isin(
            [
                "Great Britain",
                "Ireland",
                "France",
                "United States",
                "Australia",
            ]
        )
    ]
    .groupby(
        ["candidate_jurisdiction", "year"],
        dropna=False,
    )[
        ["text_prize_rows", "real_prize_rows", "integer_prize_rows"]
    ]
    .sum()
    .reset_index()
    .sort_values(["candidate_jurisdiction", "year"])
)

numeric_storage_by_year

,candidate_jurisdiction,year,text_prize_rows,real_prize_rows,integer_prize_rows
0,Australia,2015,0,3971,39
1,Australia,2016,0,2677,2
2,Australia,2017,0,2944,1
3,Australia,2018,0,3305,2
4,Australia,2019,0,3272,5
5,Australia,2020,0,2022,0
6,Australia,2021,0,3128,63
7,Australia,2022,0,3085,27
8,Australia,2023,0,1862,2
9,Australia,2024,0,3332,0


In [26]:
# Inspect race-level prize totals and winner amounts across major jurisdictions.
#
# Several foreign values resemble currency conversions into pounds rather than
# natural local-currency prize schedules. For example, repeated thirds such as
# 333333.33 and 111111.11 may result from dividing round local amounts by an
# exchange rate.
#
# This cell selects a small set of high-value races from major jurisdictions
# and reports:
# - the winner's raw prize;
# - the sum of all populated runner prizes;
# - the number of paid runners; and
# - the race description.
#
# The results will provide candidate cases for external validation. No currency
# or conversion interpretation is assigned in this cell.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    race_prize_amounts = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            MIN(race_name) AS race_name,
            MIN(type) AS type,
            MAX(
                CASE
                    WHEN CAST(pos AS TEXT) = '1'
                    THEN CAST(
                        REPLACE(
                            REPLACE(CAST(prize AS TEXT), '€', ''),
                            ',',
                            ''
                        ) AS REAL
                    )
                END
            ) AS winner_raw_prize,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> ''
                    THEN CAST(
                        REPLACE(
                            REPLACE(CAST(prize AS TEXT), '€', ''),
                            ',',
                            ''
                        ) AS REAL
                    )
                    ELSE 0
                END
            ) AS summed_runner_prizes,
            SUM(
                CASE
                    WHEN TRIM(CAST(prize AS TEXT)) <> '' THEN 1
                    ELSE 0
                END
            ) AS paid_runner_rows
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY date, course, off
        """,
        conn,
    )

race_prize_amounts = race_prize_amounts.merge(
    race_prize_profiles[
        ["date", "course", "off", "candidate_jurisdiction"]
    ],
    on=["date", "course", "off"],
    how="left",
    validate="one_to_one",
)

high_value_race_examples = (
    race_prize_amounts.loc[
        race_prize_amounts["candidate_jurisdiction"].isin(
            [
                "Great Britain",
                "Ireland",
                "France",
                "United States",
                "Australia",
            ]
        )
    ]
    .sort_values(
        ["candidate_jurisdiction", "summed_runner_prizes"],
        ascending=[True, False],
    )
    .groupby("candidate_jurisdiction", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

high_value_race_examples

,date,course,off,race_name,type,winner_raw_prize,summed_runner_prizes,paid_runner_rows,candidate_jurisdiction
0,2023-10-14,Randwick (AUS),6:15,The TAB Everest (Conditions) (3yo+) (Turf),Flat,4264971.75,11468361.60,12,Australia
1,2024-10-19,Randwick (AUS),6:15,The TAB Everest (3yo+) (Turf),Flat,3743315.51,10187165.78,11,Australia
2,2025-10-18,Randwick,06:15,The TAB Everest (2yo+) (Turf),Flat,3465346.53,9777227.70,12,Australia
3,2021-10-16,Randwick (AUS),6:15,The TAB Everest (Conditions) (2yo+) (Turf),Flat,3791573.03,8651123.59,12,Australia
4,2022-10-15,Randwick (AUS),6:15,The TAB Everest (Conditions) (3yo+) (Turf),Flat,3628494.62,8258870.97,12,Australia
5,2019-10-06,Longchamp (FR),3:05,Qatar Prix de lArc de Triomphe (3yo+) (Grande...,Flat,2573873.87,4504504.50,5,France
6,2021-10-03,Longchamp (FR),3:05,Qatar Prix de lArc de Triomphe (3yo+ No Geldi...,Flat,2550892.86,4464285.71,5,France
7,2018-10-07,Longchamp (FR),3:05,Qatar Prix de lArc de Triomphe (3yo+ No Geldi...,Flat,2528318.58,4424778.75,5,France
8,2023-10-01,Longchamp (FR),3:05,Qatar Prix de lArc de Triomphe (3yo+ Colts Ho...,Flat,2528318.58,4424778.75,5,France
9,2024-10-06,Longchamp (FR),3:20,Qatar Prix de lArc de Triomphe (3yo+ No Geldi...,Flat,2484347.83,4347826.09,5,France


## Interim structural findings

The source field is named `prize` and is declared as `INTEGER`, but SQLite actually stores a mixture of text, real and integer values.

Effective availability is:

- 1,011,570 populated runner rows out of 1,851,285;
- 189,031 provisional races with at least one populated value out of 189,043;
- 12 races with no populated value.

The field behaves primarily as a **runner-level prize allocation**, not as a race-level purse:

- populated values normally occur on placed runners;
- several runners within the same race usually receive different amounts;
- lower placings may receive identical minimum payments;
- blanks generally indicate runners receiving no recorded prize.

A small number of non-finishers also carry values. These occur across several jurisdictions and must be retained as valid source observations pending jurisdiction-specific validation.

Raw formatting is jurisdiction-dependent:

- Irish populated values are consistently stored as euro-prefixed text;
- all other populated values are stored numerically;
- numeric storage as `INTEGER` or `REAL` reflects whether the resulting value contains a fractional component, not a stable semantic distinction;
- no populated numeric value requires more than two decimal places.

High-value foreign examples contain conspicuous conversion-like decimals. The current candidate interpretation is therefore:

1. Great Britain — runner prize recorded in pounds;
2. Ireland — runner prize recorded in euros with an explicit `€` symbol;
3. at least some other jurisdictions — runner prize apparently converted into pounds by the source.

The apparent foreign-currency conversion remains a candidate interpretation and must be tested against external historical results before being confirmed.

In [27]:
# Build a small external-validation sample of prominent foreign races.
#
# These races are useful because their official prize schedules should be
# recoverable from historical racecards or governing-body results.
#
# We retain every paid runner so that any inferred exchange rate can be tested
# across the complete placing schedule, not just against the winner.
#
# No reverse conversion is attempted yet.

validation_name_patterns = [
    "Pegasus World Cup",
    "Breeders Cup Classic",
    "Prix de lArc de Triomphe",
    "The TAB Everest",
]

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    external_validation_candidates = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            race_name,
            pos,
            horse,
            prize,
            typeof(prize) AS prize_storage_class
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND TRIM(CAST(prize AS TEXT)) <> ''
          AND (
              race_name LIKE '%Pegasus World Cup%'
              OR race_name LIKE '%Breeders Cup Classic%'
              OR race_name LIKE '%Prix de lArc de Triomphe%'
              OR race_name LIKE '%The TAB Everest%'
          )
        ORDER BY
            date,
            course,
            off,
            CASE
                WHEN typeof(pos) IN ('integer', 'real') THEN pos
                ELSE 999
            END,
            rowid
        """,
        conn,
    )

external_validation_candidates[
    [
        "date",
        "course",
        "off",
        "race_name",
    ]
].drop_duplicates().reset_index(drop=True)

,date,course,off,race_name
0,2015-10-04,Longchamp (FR),2:55,Qatar Prix de lArc de Triomphe (3yo+ No Geldi...
1,2015-10-31,Keeneland (USA),9:35,Breeders Cup Classic (3yo+) (Dirt)
2,2016-10-02,Chantilly (FR),3:05,Qatar Prix de lArc de Triomphe (3yo+ No Geldi...
3,2016-11-05,Chelmsford (AW),7:20,Breeders Cup Classic Live On attheraces Handicap
4,2016-11-06,Santa Anita (USA),12:35,Breeders Cup Classic (3yo+) (Dirt)
5,2017-01-28,Gulfstream Park (USA),10:40,Pegasus World Cup Invitational Stakes (4yo+) ...
6,2017-10-01,Chantilly (FR),3:05,Qatar Prix de lArc de Triomphe (3yo+ No Geldi...
7,2017-11-05,Del Mar (USA),12:35,Breeders Cup Classic (3yo+) (Dirt)
8,2018-01-27,Gulfstream Park (USA),10:35,Pegasus World Cup Invitational Stakes (4yo+) ...
9,2018-10-07,Longchamp (FR),3:05,Qatar Prix de lArc de Triomphe (3yo+ No Geldi...


In [28]:
# Select a small, controlled set of races for external validation.
#
# The broad name search found one false positive: a Chelmsford handicap whose
# title merely referred to the Breeders' Cup broadcast. We therefore identify
# validation races by exact date + course + off, using the established
# provisional race identity.
#
# The sample covers three foreign-currency jurisdictions and several years:
# - United States;
# - France; and
# - Australia.
#
# Every runner row is retained so an inferred exchange rate can later be tested
# against the complete official prize schedule, not just the winner's amount.

validation_race_keys = pd.DataFrame(
    [
        {
            "date": "2018-01-27",
            "course": "Gulfstream Park (USA)",
            "off": "10:35",
            "validation_label": "2018 Pegasus World Cup",
        },
        {
            "date": "2025-11-01",
            "course": "Del Mar",
            "off": "22:25",
            "validation_label": "2025 Breeders Cup Classic",
        },
        {
            "date": "2019-10-06",
            "course": "Longchamp (FR)",
            "off": "3:05",
            "validation_label": "2019 Prix de lArc de Triomphe",
        },
        {
            "date": "2023-10-14",
            "course": "Randwick (AUS)",
            "off": "6:15",
            "validation_label": "2023 The Everest",
        },
    ]
)

controlled_validation_sample = (
    external_validation_candidates
    .merge(
        validation_race_keys,
        on=["date", "course", "off"],
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        ["date", "course", "off", "pos"],
        key=lambda series: pd.to_numeric(series, errors="coerce"),
        na_position="last",
    )
    .reset_index(drop=True)
)

controlled_validation_sample[
    [
        "validation_label",
        "date",
        "course",
        "off",
        "pos",
        "horse",
        "prize",
    ]
]

,validation_label,date,course,off,pos,horse,prize
0,2018 Pegasus World Cup,2018-01-27,Gulfstream Park (USA),10:35,1,Gun Runner (USA),5185185.19
1,2019 Prix de lArc de Triomphe,2019-10-06,Longchamp (FR),3:05,1,Waldgeist (GB),2573873.87
2,2023 The Everest,2023-10-14,Randwick (AUS),6:15,1,Think About It (AUS),4264971.75
3,2025 Breeders Cup Classic,2025-11-01,Del Mar,22:25,1,Forever Young (JPN),2912000.00
4,2018 Pegasus World Cup,2018-01-27,Gulfstream Park (USA),10:35,2,West Coast (USA),1185185.19
5,2019 Prix de lArc de Triomphe,2019-10-06,Longchamp (FR),3:05,2,Enable (GB),1029729.73
6,2023 The Everest,2023-10-14,Randwick (AUS),6:15,2,I Wish I Win (NZ),1638418.08
7,2025 Breeders Cup Classic,2025-11-01,Del Mar,22:25,2,Sierra Leone (USA),952000.00
8,2018 Pegasus World Cup,2018-01-27,Gulfstream Park (USA),10:35,3,Gunnevera (USA),962962.96
9,2019 Prix de lArc de Triomphe,2019-10-06,Longchamp (FR),3:05,3,Sottsass (FR),514864.86


In [29]:
# Compare the stored 2018 Pegasus World Cup values with the externally
# reported official US-dollar prize schedule.
#
# External evidence:
# - winner: $7,000,000
# - second: $1,600,000
# - third: $1,300,000
# - fourth: $1,000,000
# - fifth: $850,000
# - positions 6–12: $650,000 each
#
# Sources consulted:
# - Bleacher Report, 27 January 2018, citing NBC Sports for the allocations
# - America's Best Racing race page, reporting a $16.3 million purse
#
# For each placing, the implied conversion rate is:
#
#     official USD prize / stored source prize
#
# A stable rate across every placing would confirm that the source stored
# converted sterling amounts rather than the original US-dollar awards.

pegasus_2018_official = pd.DataFrame(
    {
        "pos": list(range(1, 13)),
        "official_local_prize": [
            7_000_000,
            1_600_000,
            1_300_000,
            1_000_000,
            850_000,
            650_000,
            650_000,
            650_000,
            650_000,
            650_000,
            650_000,
            650_000,
        ],
        "official_currency": "USD",
    }
)

pegasus_2018_source = (
    controlled_validation_sample.loc[
        controlled_validation_sample["validation_label"]
        == "2018 Pegasus World Cup",
        ["pos", "horse", "prize"],
    ]
    .copy()
)

pegasus_2018_source["pos"] = pd.to_numeric(
    pegasus_2018_source["pos"],
    errors="raise",
).astype(int)

pegasus_2018_validation = pegasus_2018_source.merge(
    pegasus_2018_official,
    on="pos",
    how="left",
    validate="one_to_one",
)

pegasus_2018_validation["implied_usd_per_stored_unit"] = (
    pegasus_2018_validation["official_local_prize"]
    / pegasus_2018_validation["prize"].astype(float)
)

pegasus_2018_validation["reconstructed_usd_at_1_35"] = (
    pegasus_2018_validation["prize"].astype(float) * 1.35
).round(2)

pegasus_2018_validation["reconstruction_difference"] = (
    pegasus_2018_validation["reconstructed_usd_at_1_35"]
    - pegasus_2018_validation["official_local_prize"]
).round(2)

pegasus_2018_validation

,pos,horse,prize,official_local_prize,official_currency,implied_usd_per_stored_unit,reconstructed_usd_at_1_35,reconstruction_difference
0,1,Gun Runner (USA),5185185.19,7000000,USD,1.35,7000000.01,0.01
1,2,West Coast (USA),1185185.19,1600000,USD,1.35,1600000.01,0.01
2,3,Gunnevera (USA),962962.96,1300000,USD,1.35,1300000.00,0.00
3,4,Fear The Cowboy (USA),740740.74,1000000,USD,1.35,1000000.00,0.00
4,5,Seeking The Soul (USA),629629.63,850000,USD,1.35,850000.00,0.00
5,6,Stellar Wind (USA),481481.48,650000,USD,1.35,650000.00,0.00
6,7,Collected (USA),481481.48,650000,USD,1.35,650000.00,0.00
7,8,Sharp Azteca (USA),481481.48,650000,USD,1.35,650000.00,0.00
8,9,Giant Expectations (USA),481481.48,650000,USD,1.35,650000.00,0.00
9,10,War Story (USA),481481.48,650000,USD,1.35,650000.00,0.00


In [30]:
# Test the 2019 Prix de l'Arc de Triomphe against its apparent official
# euro-denominated placing schedule.
#
# The source contains five paid runners whose stored values appear to reconstruct
# round or half-round euro amounts under one common conversion rate.
#
# These candidate official allocations sum to the advertised €5 million purse:
# - first:  €2,857,000
# - second: €1,143,000
# - third:    €571,500
# - fourth:   €285,500
# - fifth:    €143,000
#
# This cell calculates the implied euro-per-stored-unit rate for each placing.
# A stable rate would confirm race-level conversion, although the exact official
# allocation schedule should still be retained as externally validated evidence.

arc_2019_official = pd.DataFrame(
    {
        "pos": [1, 2, 3, 4, 5],
        "official_local_prize": [
            2_857_000,
            1_143_000,
            571_500,
            285_500,
            143_000,
        ],
        "official_currency": "EUR",
    }
)

arc_2019_source = (
    controlled_validation_sample.loc[
        controlled_validation_sample["validation_label"]
        == "2019 Prix de lArc de Triomphe",
        ["pos", "horse", "prize"],
    ]
    .copy()
)

arc_2019_source["pos"] = pd.to_numeric(
    arc_2019_source["pos"],
    errors="raise",
).astype(int)

arc_2019_validation = arc_2019_source.merge(
    arc_2019_official,
    on="pos",
    how="left",
    validate="one_to_one",
)

arc_2019_validation["implied_eur_per_stored_unit"] = (
    arc_2019_validation["official_local_prize"]
    / arc_2019_validation["prize"].astype(float)
)

arc_2019_validation

,pos,horse,prize,official_local_prize,official_currency,implied_eur_per_stored_unit
0,1,Waldgeist (GB),2573873.87,2857000,EUR,1.11
1,2,Enable (GB),1029729.73,1143000,EUR,1.11
2,3,Sottsass (FR),514864.86,571500,EUR,1.11
3,4,Japan (GB),257207.21,285500,EUR,1.11
4,5,Magical (IRE),128828.83,143000,EUR,1.11


In [31]:
# Summarise the two confirmed source conversion examples.
#
# Each validation has shown one fixed local-currency-per-stored-unit rate across
# every paid placing in the race. We now calculate race-level diagnostics:
# - minimum and maximum implied rate;
# - spread between those rates;
# - maximum reconstruction error after rounding to two decimals; and
# - the number of placings tested.
#
# A near-zero rate spread confirms that conversion was applied consistently
# across the complete runner-level prize schedule.

conversion_validation_summary = pd.DataFrame(
    [
        {
            "validation_race": "2018 Pegasus World Cup",
            "local_currency": "USD",
            "tested_placings": len(pegasus_2018_validation),
            "minimum_implied_rate":
                pegasus_2018_validation[
                    "implied_usd_per_stored_unit"
                ].min(),
            "maximum_implied_rate":
                pegasus_2018_validation[
                    "implied_usd_per_stored_unit"
                ].max(),
            "maximum_absolute_reconstruction_error":
                pegasus_2018_validation[
                    "reconstruction_difference"
                ].abs().max(),
        },
        {
            "validation_race": "2019 Prix de lArc de Triomphe",
            "local_currency": "EUR",
            "tested_placings": len(arc_2019_validation),
            "minimum_implied_rate":
                arc_2019_validation[
                    "implied_eur_per_stored_unit"
                ].min(),
            "maximum_implied_rate":
                arc_2019_validation[
                    "implied_eur_per_stored_unit"
                ].max(),
            "maximum_absolute_reconstruction_error": (
                (
                    arc_2019_validation["prize"].astype(float) * 1.11
                    - arc_2019_validation["official_local_prize"]
                )
                .abs()
                .max()
            ),
        },
    ]
)

conversion_validation_summary["implied_rate_spread"] = (
    conversion_validation_summary["maximum_implied_rate"]
    - conversion_validation_summary["minimum_implied_rate"]
)

conversion_validation_summary.round(8)

,validation_race,local_currency,tested_placings,minimum_implied_rate,maximum_implied_rate,maximum_absolute_reconstruction_error,implied_rate_spread
0,2018 Pegasus World Cup,USD,12,1.35,1.35,0.0100,1.000000e-08
1,2019 Prix de lArc de Triomphe,EUR,5,1.11,1.11,0.0054,2.000000e-08


In [32]:
# Examine whether foreign races on the same date appear to share one source
# exchange rate.
#
# Exact reverse engineering requires an official local-currency amount, which
# we do not have for every race. However, converted prize schedules often reveal
# a common rate because several stored values reconstruct to plausible round
# local amounts under the same multiplier.
#
# We start with all US races on the 2018 Pegasus date. The output retains each
# race's distinct populated prize values so we can test whether multiplying by
# 1.35 reconstructs consistently round dollar amounts beyond the validation race.
#
# This is an exploratory diagnostic, not yet a global conversion rule.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    same_day_us_prizes = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            race_name,
            pos,
            horse,
            CAST(prize AS REAL) AS stored_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date = '2018-01-27'
          AND course = 'Gulfstream Park (USA)'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        ORDER BY
            off,
            CASE
                WHEN typeof(pos) IN ('integer', 'real') THEN pos
                ELSE 999
            END,
            rowid
        """,
        conn,
    )

same_day_us_prizes["reconstructed_usd_at_1_35"] = (
    same_day_us_prizes["stored_prize"] * 1.35
).round(2)

same_day_us_prizes[
    [
        "off",
        "race_name",
        "pos",
        "stored_prize",
        "reconstructed_usd_at_1_35",
    ]
]

,off,race_name,pos,stored_prize,reconstructed_usd_at_1_35
0,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,1,5185185.19,7000000.01
1,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,2,1185185.19,1600000.01
2,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,3,962962.96,1300000.00
3,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,4,740740.74,1000000.00
4,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,5,629629.63,850000.00
5,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,6,481481.48,650000.00
6,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,7,481481.48,650000.00
7,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,8,481481.48,650000.00
8,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,9,481481.48,650000.00
9,10:35,Pegasus World Cup Invitational Stakes (4yo+) ...,10,481481.48,650000.00


In [33]:
# Quantify how cleanly the 1.35 multiplier reconstructs dollar amounts across
# the entire Gulfstream Park card on 27 January 2018.
#
# For each populated prize value, we calculate:
# - the reconstructed US-dollar amount;
# - its distance from the nearest cent; and
# - its distance from the nearest whole dollar.
#
# Near-zero cent residuals across the full card would show that one shared
# conversion rate was systematically applied to every race at the meeting.

same_day_us_prizes["reconstructed_usd"] = (
    same_day_us_prizes["stored_prize"] * 1.35
)

same_day_us_prizes["distance_from_nearest_cent"] = (
    same_day_us_prizes["reconstructed_usd"]
    - same_day_us_prizes["reconstructed_usd"].round(2)
).abs()

same_day_us_prizes["distance_from_nearest_dollar"] = (
    same_day_us_prizes["reconstructed_usd"]
    - same_day_us_prizes["reconstructed_usd"].round(0)
).abs()

same_day_rate_diagnostics = pd.DataFrame(
    {
        "measure": [
            "populated prize rows",
            "provisional races",
            "maximum distance from nearest cent",
            "rows within one cent of a whole dollar",
            "rows within one cent of a 25-cent increment",
        ],
        "value": [
            len(same_day_us_prizes),
            same_day_us_prizes[["off", "race_name"]].drop_duplicates().shape[0],
            same_day_us_prizes["distance_from_nearest_cent"].max(),
            (
                same_day_us_prizes["distance_from_nearest_dollar"]
                <= 0.01
            ).sum(),
            (
                (
                    same_day_us_prizes["reconstructed_usd"] * 4
                    - (same_day_us_prizes["reconstructed_usd"] * 4).round()
                ).abs()
                <= 0.01
            ).sum(),
        ],
    }
)

same_day_rate_diagnostics

,measure,value
0,populated prize rows,60.000
1,provisional races,6.000
2,maximum distance from nearest cent,0.005
3,rows within one cent of a whole dollar,60.000
4,rows within one cent of a 25-cent increment,24.000


In [34]:
# Check whether the same inferred USD conversion rate applies to other US
# meetings on the same calendar date.
#
# If 1.35 reconstructs plausible whole-dollar amounts at other US courses on
# 27 January 2018, that would suggest a date-wide source exchange rate.
# If it works only at Gulfstream Park, the rate may instead be attached to a
# meeting, feed, or separately processed racecard.
#
# We therefore load every populated US prize row on that date and measure how
# closely multiplication by 1.35 lands on whole-dollar amounts by course.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    same_date_us_prizes = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            race_name,
            pos,
            CAST(prize AS REAL) AS stored_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date = '2018-01-27'
          AND course LIKE '%(USA)%'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        """,
        conn,
    )

same_date_us_prizes["reconstructed_usd"] = (
    same_date_us_prizes["stored_prize"] * 1.35
)

same_date_us_prizes["distance_from_nearest_dollar"] = (
    same_date_us_prizes["reconstructed_usd"]
    - same_date_us_prizes["reconstructed_usd"].round()
).abs()

same_date_us_rate_profile = (
    same_date_us_prizes
    .groupby("course", dropna=False)
    .agg(
        provisional_races=("off", "nunique"),
        populated_prize_rows=("stored_prize", "size"),
        rows_within_one_cent_of_whole_dollar=(
            "distance_from_nearest_dollar",
            lambda values: (values <= 0.01).sum(),
        ),
        maximum_distance_from_whole_dollar=(
            "distance_from_nearest_dollar",
            "max",
        ),
    )
    .reset_index()
    .sort_values("populated_prize_rows", ascending=False)
)

same_date_us_rate_profile

,course,provisional_races,populated_prize_rows,rows_within_one_cent_of_whole_dollar,maximum_distance_from_whole_dollar
1,Gulfstream Park (USA),6,60,60,0.0065
0,Aqueduct (USA),1,6,6,0.0065


In [35]:
# Test whether the inferred USD conversion rate changes between adjacent dates.
#
# We compare all populated US prize values on 27 and 28 January 2018.
# For each date, we search a narrow range of plausible USD-per-GBP rates and
# select the rate that makes the largest number of reconstructed values land
# within one cent of a whole US dollar.
#
# This is an exploratory reverse-engineering method. A strong daily optimum
# would indicate that the source used a date-level currency rate, while the
# same optimum across both dates could indicate a less frequently updated rate.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    adjacent_date_us_prizes = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            CAST(prize AS REAL) AS stored_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date IN ('2018-01-27', '2018-01-28')
          AND course LIKE '%(USA)%'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        """,
        conn,
    )

candidate_rates = np.round(np.arange(1.20, 1.5001, 0.01), 2)

rate_search_rows = []

for race_date, group in adjacent_date_us_prizes.groupby("date"):
    stored_values = group["stored_prize"].to_numpy(dtype=float)

    for candidate_rate in candidate_rates:
        reconstructed = stored_values * candidate_rate
        distance_from_whole_dollar = np.abs(
            reconstructed - np.round(reconstructed)
        )

        rate_search_rows.append(
            {
                "date": race_date,
                "candidate_rate": candidate_rate,
                "populated_prize_rows": len(stored_values),
                "rows_within_one_cent_of_whole_dollar":
                    int((distance_from_whole_dollar <= 0.01).sum()),
                "maximum_distance_from_whole_dollar":
                    distance_from_whole_dollar.max(),
                "mean_distance_from_whole_dollar":
                    distance_from_whole_dollar.mean(),
            }
        )

daily_rate_search = pd.DataFrame(rate_search_rows)

best_daily_rates = (
    daily_rate_search
    .sort_values(
        [
            "date",
            "rows_within_one_cent_of_whole_dollar",
            "mean_distance_from_whole_dollar",
        ],
        ascending=[True, False, True],
    )
    .groupby("date", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

best_daily_rates

,date,candidate_rate,populated_prize_rows,rows_within_one_cent_of_whole_dollar,maximum_distance_from_whole_dollar,mean_distance_from_whole_dollar
0,2018-01-27,1.35,66,66,0.0065,0.003788
1,2018-01-27,1.26,66,17,0.4718,0.251939
2,2018-01-27,1.44,66,17,0.4624,0.252873
3,2018-01-27,1.23,66,2,0.4490,0.222318
4,2018-01-27,1.50,66,2,0.4500,0.225152
5,2018-01-28,1.35,10,10,0.0060,0.002650
6,2018-01-28,1.44,10,6,0.3344,0.134240
7,2018-01-28,1.26,10,6,0.3380,0.135620
8,2018-01-28,1.45,10,0,0.4380,0.193450
9,2018-01-28,1.25,10,0,0.4500,0.198750


In [36]:
# Test the apparent USD conversion regime across January 2018.
#
# For each race date containing US prize data, we compare candidate exchange
# rates from 1.20 to 1.50 in increments of 0.01.
#
# The best candidate is the rate that makes the largest number of stored prize
# values reconstruct to within one cent of a whole US-dollar amount.
#
# This will show whether 1.35 was:
# - specific to the Pegasus weekend;
# - stable across a longer period; or
# - replaced by different rounded rates during the month.
#
# The inferred rates remain source-transformation candidates rather than
# independently verified historical market exchange rates.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    january_2018_us_prizes = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            CAST(prize AS REAL) AS stored_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date >= '2018-01-01'
          AND date < '2018-02-01'
          AND course LIKE '%(USA)%'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        """,
        conn,
    )

candidate_rates = np.round(np.arange(1.20, 1.5001, 0.01), 2)

daily_best_rate_rows = []

for race_date, group in january_2018_us_prizes.groupby("date"):
    stored_values = group["stored_prize"].to_numpy(dtype=float)
    candidate_results = []

    for candidate_rate in candidate_rates:
        reconstructed = stored_values * candidate_rate
        residuals = np.abs(reconstructed - np.round(reconstructed))

        candidate_results.append(
            {
                "candidate_rate": candidate_rate,
                "matching_rows": int((residuals <= 0.01).sum()),
                "mean_residual": residuals.mean(),
                "maximum_residual": residuals.max(),
            }
        )

    best_candidate = sorted(
        candidate_results,
        key=lambda row: (
            -row["matching_rows"],
            row["mean_residual"],
        ),
    )[0]

    daily_best_rate_rows.append(
        {
            "date": race_date,
            "courses": group["course"].nunique(),
            "provisional_races": (
                group[["course", "off"]].drop_duplicates().shape[0]
            ),
            "populated_prize_rows": len(group),
            "best_candidate_rate": best_candidate["candidate_rate"],
            "matching_rows": best_candidate["matching_rows"],
            "matching_pct": round(
                100 * best_candidate["matching_rows"] / len(group),
                2,
            ),
            "maximum_residual": best_candidate["maximum_residual"],
        }
    )

january_2018_daily_rates = (
    pd.DataFrame(daily_best_rate_rows)
    .sort_values("date")
    .reset_index(drop=True)
)

january_2018_daily_rates

,date,courses,provisional_races,populated_prize_rows,best_candidate_rate,matching_rows,matching_pct,maximum_residual
0,2018-01-06,1,2,16,1.23,16,100.0,0.0060
1,2018-01-07,1,1,8,1.35,8,100.0,0.0060
2,2018-01-13,3,6,48,1.35,48,100.0,0.0065
3,2018-01-15,1,1,7,1.35,7,100.0,0.0065
4,2018-01-20,1,1,7,1.35,7,100.0,0.0065
5,2018-01-27,2,7,66,1.35,66,100.0,0.0065
6,2018-01-28,1,2,10,1.35,10,100.0,0.0060


In [37]:
# Inspect the US prize values from 6 January 2018.
#
# The rate search found that multiplying every stored value by 1.23 produces
# whole-dollar amounts, while later January dates fit 1.35.
#
# We display the reconstructed amounts under both candidate rates so we can see
# whether:
# - 1.23 clearly restores a natural official prize schedule;
# - 1.35 also produces plausible amounts by coincidence; or
# - these races follow a different source convention.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    january_6_us_prizes = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            race_name,
            pos,
            horse,
            CAST(prize AS REAL) AS stored_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date = '2018-01-06'
          AND course LIKE '%(USA)%'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        ORDER BY
            course,
            off,
            CASE
                WHEN typeof(pos) IN ('integer', 'real') THEN pos
                ELSE 999
            END,
            rowid
        """,
        conn,
    )

january_6_us_prizes["reconstructed_usd_at_1_23"] = (
    january_6_us_prizes["stored_prize"] * 1.23
).round(2)

january_6_us_prizes["reconstructed_usd_at_1_35"] = (
    january_6_us_prizes["stored_prize"] * 1.35
).round(2)

january_6_us_prizes[
    [
        "course",
        "off",
        "race_name",
        "pos",
        "stored_prize",
        "reconstructed_usd_at_1_23",
        "reconstructed_usd_at_1_35",
    ]
]

,course,off,race_name,pos,stored_prize,reconstructed_usd_at_1_23,reconstructed_usd_at_1_35
0,Santa Anita (USA),10:30,Sham Stakes (3yo) (Dirt),1,48780.49,60000.00,65853.66
1,Santa Anita (USA),10:30,Sham Stakes (3yo) (Dirt),2,16260.16,20000.00,21951.22
2,Santa Anita (USA),10:30,Sham Stakes (3yo) (Dirt),3,9756.10,12000.00,13170.74
3,Santa Anita (USA),10:30,Sham Stakes (3yo) (Dirt),4,4878.05,6000.00,6585.37
4,Santa Anita (USA),10:30,Sham Stakes (3yo) (Dirt),5,1626.02,2000.00,2195.13
5,Santa Anita (USA),10:30,Sham Stakes (3yo) (Dirt),6,280.49,345.00,378.66
6,Santa Anita (USA),11:30,San Gabriel Stakes (4yo+) (Turf),1,97560.98,120000.01,131707.32
7,Santa Anita (USA),11:30,San Gabriel Stakes (4yo+) (Turf),2,32520.33,40000.01,43902.45
8,Santa Anita (USA),11:30,San Gabriel Stakes (4yo+) (Turf),3,19512.20,24000.01,26341.47
9,Santa Anita (USA),11:30,San Gabriel Stakes (4yo+) (Turf),4,9756.10,12000.00,13170.74


### Source conversion-rate warning

The inferred conversion multiplier must not automatically be interpreted as the historical market exchange rate on the race date.

For United States races on 6 January 2018, multiplying the stored values by `1.23` reconstructs coherent dollar prize schedules exactly. From 7 January onward, the sampled January races reconstruct under `1.35`.

The abrupt change does not establish that the foreign-exchange market moved between those dates. It may instead reflect:

- a stale source conversion rate;
- a periodically updated lookup table;
- conversion performed on a publication or ingestion date;
- different upstream data batches; or
- an erroneous source rate.

The reconstructed local amounts may therefore be recoverable even when the source conversion rate is not historically appropriate. Any reverse-engineering model must distinguish:

1. the **inferred source conversion multiplier**;
2. the **official original local-currency amount**; and
3. the **historical market exchange rate**, which is a separate downstream reference.

In [38]:
# Infer the best-fitting USD conversion multiplier for every US race date in 2018.
#
# For each date, candidate rates from 1.10 to 1.60 are tested in increments
# of 0.01. The selected rate is the one that makes the largest number of stored
# prize values reconstruct to within one cent of a whole US-dollar amount.
#
# We retain diagnostics rather than treating the result as confirmed:
# - the number and percentage of matching prize rows;
# - the number of courses and races supporting the inference; and
# - the gap between the best and second-best candidate rates.
#
# A large gap and 100% matching provide strong internal evidence for the
# source multiplier. Weak or ambiguous dates must remain unresolved.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    us_2018_prizes = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            CAST(prize AS REAL) AS stored_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date >= '2018-01-01'
          AND date < '2019-01-01'
          AND course LIKE '%(USA)%'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        """,
        conn,
    )

candidate_rates = np.round(np.arange(1.10, 1.6001, 0.01), 2)

daily_rate_results = []

for race_date, group in us_2018_prizes.groupby("date"):
    stored_values = group["stored_prize"].to_numpy(dtype=float)
    candidates = []

    for candidate_rate in candidate_rates:
        reconstructed = stored_values * candidate_rate
        residuals = np.abs(reconstructed - np.round(reconstructed))

        candidates.append(
            {
                "candidate_rate": candidate_rate,
                "matching_rows": int((residuals <= 0.01).sum()),
                "mean_residual": residuals.mean(),
            }
        )

    ranked = sorted(
        candidates,
        key=lambda row: (
            -row["matching_rows"],
            row["mean_residual"],
        ),
    )

    best = ranked[0]
    second = ranked[1]

    daily_rate_results.append(
        {
            "date": race_date,
            "courses": group["course"].nunique(),
            "provisional_races": (
                group[["course", "off"]].drop_duplicates().shape[0]
            ),
            "populated_prize_rows": len(group),
            "best_candidate_rate": best["candidate_rate"],
            "matching_rows": best["matching_rows"],
            "matching_pct": round(
                100 * best["matching_rows"] / len(group),
                2,
            ),
            "second_best_matching_rows": second["matching_rows"],
            "matching_row_advantage": (
                best["matching_rows"] - second["matching_rows"]
            ),
        }
    )

us_2018_daily_rate_profile = (
    pd.DataFrame(daily_rate_results)
    .sort_values("date")
    .reset_index(drop=True)
)

us_2018_daily_rate_profile

,date,courses,provisional_races,populated_prize_rows,best_candidate_rate,matching_rows,matching_pct,second_best_matching_rows,matching_row_advantage
0,2018-01-06,1,2,16,1.23,16,100.0,1,15
1,2018-01-07,1,1,8,1.35,8,100.0,6,2
2,2018-01-13,3,6,48,1.35,48,100.0,31,17
3,2018-01-15,1,1,7,1.35,7,100.0,5,2
4,2018-01-20,1,1,7,1.35,7,100.0,5,2
...,...,...,...,...,...,...,...,...,...
165,2018-12-20,1,1,6,1.35,6,100.0,3,3
166,2018-12-22,1,1,11,1.35,11,100.0,1,10
167,2018-12-26,1,4,41,1.35,41,100.0,39,2
168,2018-12-27,1,1,5,1.17,5,100.0,5,0


In [39]:
# Summarise the strength and frequency of inferred US conversion rates in 2018.
#
# A date is classified as:
# - strong: every prize row matches and the best rate beats the second-best
#   candidate by at least 20% of the populated rows;
# - moderate: every row matches and the best rate has some advantage;
# - ambiguous: every row matches but another candidate performs equally well;
# - incomplete: the best candidate does not reconstruct every row.
#
# This prevents a mathematically possible multiplier from being treated as a
# confirmed source rate when several rates fit the same round prize schedule.

us_2018_daily_rate_profile["inference_strength"] = np.select(
    [
        (
            (us_2018_daily_rate_profile["matching_pct"] == 100)
            & (
                us_2018_daily_rate_profile["matching_row_advantage"]
                >= 0.20
                * us_2018_daily_rate_profile["populated_prize_rows"]
            )
        ),
        (
            (us_2018_daily_rate_profile["matching_pct"] == 100)
            & (us_2018_daily_rate_profile["matching_row_advantage"] > 0)
        ),
        (
            (us_2018_daily_rate_profile["matching_pct"] == 100)
            & (us_2018_daily_rate_profile["matching_row_advantage"] == 0)
        ),
    ],
    [
        "strong",
        "moderate",
        "ambiguous",
    ],
    default="incomplete",
)

us_2018_rate_summary = (
    us_2018_daily_rate_profile
    .groupby(
        ["best_candidate_rate", "inference_strength"],
        dropna=False,
    )
    .agg(
        race_dates=("date", "size"),
        provisional_races=("provisional_races", "sum"),
        populated_prize_rows=("populated_prize_rows", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
    .sort_values(
        ["race_dates", "best_candidate_rate"],
        ascending=[False, True],
    )
)

us_2018_rate_summary

,best_candidate_rate,inference_strength,race_dates,provisional_races,populated_prize_rows,first_date,last_date
4,1.35,strong,143,526,4128,2018-01-07,2018-12-22
0,1.17,ambiguous,12,14,98,2018-02-25,2018-12-29
3,1.35,moderate,11,31,253,2018-04-01,2018-12-26
2,1.35,incomplete,3,16,120,2018-05-12,2018-11-22
1,1.23,strong,1,2,16,2018-01-06,2018-01-06


In [40]:
# Inspect the three US dates where no candidate rate reconstructed every prize row.
#
# These dates may contain:
# - one or more source errors;
# - a mixture of conversion rates;
# - non-round original prize amounts;
# - or an inadequately narrow candidate-rate search.
#
# We compare each row with the dominant 1.35 multiplier and retain race identity
# so any anomaly can be traced to a specific race or placing.

incomplete_us_dates = (
    us_2018_daily_rate_profile.loc[
        us_2018_daily_rate_profile["inference_strength"] == "incomplete",
        "date",
    ]
    .tolist()
)

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    incomplete_us_rows = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            race_name,
            pos,
            horse,
            CAST(prize AS REAL) AS stored_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND date IN ({",".join("?" for _ in incomplete_us_dates)})
          AND course LIKE '%(USA)%'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        ORDER BY
            date,
            course,
            off,
            CASE
                WHEN typeof(pos) IN ('integer', 'real') THEN pos
                ELSE 999
            END,
            rowid
        """,
        conn,
        params=incomplete_us_dates,
    )

incomplete_us_rows["reconstructed_usd_at_1_35"] = (
    incomplete_us_rows["stored_prize"] * 1.35
).round(2)

incomplete_us_rows["distance_from_nearest_dollar"] = (
    incomplete_us_rows["reconstructed_usd_at_1_35"]
    - incomplete_us_rows["reconstructed_usd_at_1_35"].round()
).abs()

incomplete_us_rows.loc[
    incomplete_us_rows["distance_from_nearest_dollar"] > 0.01,
    [
        "date",
        "course",
        "off",
        "race_name",
        "pos",
        "horse",
        "stored_prize",
        "reconstructed_usd_at_1_35",
        "distance_from_nearest_dollar",
    ],
]

,date,course,off,race_name,pos,horse,stored_prize,reconstructed_usd_at_1_35,distance_from_nearest_dollar
3,2018-05-12,Belmont Park (USA),10:14,Runhappy Stakes () (4yo+) (Dirt),4,Great Stuff (USA),5555.56,7500.01,0.01
9,2018-05-12,Belmont Park (USA),10:46,Beaugay Stakes (4yo+ Fillies & Mares) (Turf),5,Tricky Escape (USA),4444.44,5999.99,0.01
12,2018-05-12,Belmont Park (USA),11:18,Peter Pan Stakes (3yo) (Dirt),1,Blended Citizen (USA),155555.56,210000.01,0.01
14,2018-05-12,Belmont Park (USA),11:18,Peter Pan Stakes (3yo) (Dirt),3,Just Whistle (USA),25925.93,35000.01,0.01
17,2018-05-12,Belmont Park (USA),11:18,Peter Pan Stakes (3yo) (Dirt),6,Gotta Go (USA),5185.19,7000.01,0.01
21,2018-05-12,Belmont Park (USA),11:50,Man O War Stakes (4yo+) (Turf),4,Call Provision (USA),34074.07,45999.99,0.01
30,2018-05-12,Belmont Park (USA),7:34,Vagrancy Handicap (4yo+ Fillies & Mares) (Dirt),5,Swing And Sway (USA),4444.44,5999.99,0.01
34,2018-05-12,Percy Warner Park (USA),11:28,Calvin Houghland Iroquois Stakes Hurdle),3,Jamarjo (IRE),14818.81,20005.39,0.39
36,2018-05-12,Percy Warner Park (USA),11:28,Calvin Houghland Iroquois Stakes Hurdle),5,Sempre Medici (FR),5925.93,8000.01,0.01
37,2018-05-12,Santa Anita (USA),10:40,Lazaro Barrera Stakes (3yo) (Dirt),1,Kanthaka (USA),44444.44,59999.99,0.01


In [41]:
# Reclassify the apparent 1.35 conversion failures using the source's
# two-decimal rounding limits.
#
# A stored value rounded to the nearest penny can produce a reconstructed
# local-currency amount slightly more than one cent from a whole dollar.
# We therefore allow a tolerance of 0.011 rather than exactly 0.01.
#
# Rows outside that tolerance are retained as substantive anomalies rather
# than ordinary conversion-and-rounding effects.

incomplete_us_rows["raw_reconstructed_usd_at_1_35"] = (
    incomplete_us_rows["stored_prize"] * 1.35
)

incomplete_us_rows["distance_from_nearest_dollar"] = (
    incomplete_us_rows["raw_reconstructed_usd_at_1_35"]
    - incomplete_us_rows["raw_reconstructed_usd_at_1_35"].round()
).abs()

substantive_conversion_anomalies = (
    incomplete_us_rows.loc[
        incomplete_us_rows["distance_from_nearest_dollar"] > 0.011,
        [
            "date",
            "course",
            "off",
            "race_name",
            "pos",
            "horse",
            "stored_prize",
            "raw_reconstructed_usd_at_1_35",
            "distance_from_nearest_dollar",
        ],
    ]
    .sort_values(
        ["date", "course", "off", "pos"]
    )
    .reset_index(drop=True)
)

substantive_conversion_anomalies

,date,course,off,race_name,pos,horse,stored_prize,raw_reconstructed_usd_at_1_35,distance_from_nearest_dollar
0,2018-05-12,Percy Warner Park (USA),11:28,Calvin Houghland Iroquois Stakes Hurdle),3,Jamarjo (IRE),14818.81,20005.3935,0.3935
1,2018-08-30,Saratoga (USA),8:52,P. G. Johnson Stakes () (Turf),3,Chocolate Kisses (USA),8888.88,11999.9880,0.0120
2,2018-11-22,Del Mar (USA),9:30,Red Carpet Handicap (3yo+ Fillies & Mares) (T...,5,Pantsonfire (IRE),1481.49,2000.0115,0.0115


In [42]:
# Apply a tolerance justified by two-decimal source rounding.
#
# With a source multiplier of 1.35, rounding the stored sterling amount to the
# nearest penny can create a reconstruction error of up to roughly 0.00675 USD.
# We use a slightly wider 0.02-dollar tolerance to avoid misclassifying harmless
# boundary effects while still exposing genuine anomalies.
#
# This cell profiles all 2018 US prize rows under the dominant 1.35 multiplier
# and isolates only rows whose reconstructed amount is not close to a whole
# dollar.

us_2018_prizes["reconstructed_usd_at_1_35"] = (
    us_2018_prizes["stored_prize"] * 1.35
)

us_2018_prizes["distance_from_nearest_dollar"] = (
    us_2018_prizes["reconstructed_usd_at_1_35"]
    - us_2018_prizes["reconstructed_usd_at_1_35"].round()
).abs()

us_2018_conversion_profile = pd.DataFrame(
    {
        "measure": [
            "populated US prize rows in 2018",
            "rows within 0.02 USD of a whole dollar",
            "rows outside 0.02 USD",
            "coverage under dominant 1.35 multiplier (%)",
        ],
        "value": [
            len(us_2018_prizes),
            (
                us_2018_prizes["distance_from_nearest_dollar"]
                <= 0.02
            ).sum(),
            (
                us_2018_prizes["distance_from_nearest_dollar"]
                > 0.02
            ).sum(),
            round(
                100
                * (
                    us_2018_prizes["distance_from_nearest_dollar"]
                    <= 0.02
                ).sum()
                / len(us_2018_prizes),
                4,
            ),
        ],
    }
)

us_2018_conversion_profile

,measure,value
0,populated US prize rows in 2018,4615.0000
1,rows within 0.02 USD of a whole dollar,4598.0000
2,rows outside 0.02 USD,17.0000
3,coverage under dominant 1.35 multiplier (%),99.6316


In [43]:
# Inspect every 2018 US prize row that does not fit the dominant 1.35
# conversion multiplier within the justified 0.02-dollar tolerance.
#
# The expected exception set is:
# - 16 rows from Santa Anita on 6 January 2018, which reconstruct cleanly
#   under a different multiplier of 1.23; and
# - one residual anomaly from the Iroquois Stakes.
#
# Confirming that exact composition will distinguish a coherent alternative
# conversion batch from isolated source-level errors.

us_2018_rate_exceptions = (
    us_2018_prizes.loc[
        us_2018_prizes["distance_from_nearest_dollar"] > 0.02
    ]
    .copy()
)

us_2018_rate_exceptions["reconstructed_usd_at_1_23"] = (
    us_2018_rate_exceptions["stored_prize"] * 1.23
)

us_2018_rate_exceptions["distance_from_whole_dollar_at_1_23"] = (
    us_2018_rate_exceptions["reconstructed_usd_at_1_23"]
    - us_2018_rate_exceptions["reconstructed_usd_at_1_23"].round()
).abs()

us_2018_rate_exceptions[
    [
        "date",
        "course",
        "off",
        "stored_prize",
        "reconstructed_usd_at_1_35",
        "distance_from_nearest_dollar",
        "reconstructed_usd_at_1_23",
        "distance_from_whole_dollar_at_1_23",
    ]
].sort_values(
    ["date", "course", "off", "stored_prize"],
    ascending=[True, True, True, False],
)

,date,course,off,stored_prize,reconstructed_usd_at_1_35,distance_from_nearest_dollar,reconstructed_usd_at_1_23,distance_from_whole_dollar_at_1_23
15,2018-01-06,Santa Anita (USA),10:30,48780.49,65853.6615,0.3385,60000.0027,0.0027
1,2018-01-06,Santa Anita (USA),10:30,16260.16,21951.2160,0.2160,19999.9968,0.0032
2,2018-01-06,Santa Anita (USA),10:30,9756.10,13170.7350,0.2650,12000.0030,0.0030
3,2018-01-06,Santa Anita (USA),10:30,4878.05,6585.3675,0.3675,6000.0015,0.0015
4,2018-01-06,Santa Anita (USA),10:30,1626.02,2195.1270,0.1270,2000.0046,0.0046
5,2018-01-06,Santa Anita (USA),10:30,280.49,378.6615,0.3385,345.0027,0.0027
6,2018-01-06,Santa Anita (USA),11:30,97560.98,131707.3230,0.3230,120000.0054,0.0054
7,2018-01-06,Santa Anita (USA),11:30,32520.33,43902.4455,0.4455,40000.0059,0.0059
8,2018-01-06,Santa Anita (USA),11:30,19512.20,26341.4700,0.4700,24000.0060,0.0060
9,2018-01-06,Santa Anita (USA),11:30,9756.10,13170.7350,0.2650,12000.0030,0.0030


In [44]:
# Summarise the recoverability of 2018 US prize values.
#
# Rows are classified as:
# - dominant_1_35: reconstruct to a whole US dollar within $0.02 at 1.35;
# - alternative_1_23: fail 1.35 but reconstruct within $0.02 at 1.23;
# - unresolved: fit neither candidate multiplier.
#
# This is a reconstruction audit of the source transformation. It does not
# claim that either multiplier was the correct historical market exchange rate.

us_2018_prizes["reconstruction_status"] = np.select(
    [
        us_2018_prizes["distance_from_nearest_dollar"] <= 0.02,
        (
            np.abs(
                us_2018_prizes["stored_prize"] * 1.23
                - np.round(us_2018_prizes["stored_prize"] * 1.23)
            )
            <= 0.02
        ),
    ],
    [
        "dominant_1_35",
        "alternative_1_23",
    ],
    default="unresolved",
)

us_2018_reconstruction_summary = (
    us_2018_prizes
    .groupby("reconstruction_status", dropna=False)
    .agg(
        runner_rows=("stored_prize", "size"),
        provisional_races=("off", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

us_2018_reconstruction_summary["coverage_pct"] = (
    100
    * us_2018_reconstruction_summary["runner_rows"]
    / len(us_2018_prizes)
).round(4)

us_2018_reconstruction_summary

,reconstruction_status,runner_rows,provisional_races,first_date,last_date,coverage_pct
0,alternative_1_23,16,2,2018-01-06,2018-01-06,0.3467
1,dominant_1_35,4598,254,2018-01-07,2018-12-29,99.6316
2,unresolved,1,1,2018-05-12,2018-05-12,0.0217


In [45]:
# Correct the provisional-race counts in the 2018 US reconstruction summary.
#
# The previous summary used the number of distinct `off` values, which is not
# a valid race count because off-times repeat across dates and courses.
#
# We now count exact provisional race identities using:
#
#     date + course + off
#
# Runner-row coverage is unchanged.

us_2018_prizes["provisional_race_key"] = (
    us_2018_prizes["date"].astype(str)
    + "|"
    + us_2018_prizes["course"].astype(str)
    + "|"
    + us_2018_prizes["off"].astype(str)
)

us_2018_reconstruction_summary = (
    us_2018_prizes
    .groupby("reconstruction_status", dropna=False)
    .agg(
        runner_rows=("stored_prize", "size"),
        provisional_races=("provisional_race_key", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

us_2018_reconstruction_summary["runner_row_coverage_pct"] = (
    100
    * us_2018_reconstruction_summary["runner_rows"]
    / len(us_2018_prizes)
).round(4)

us_2018_reconstruction_summary

,reconstruction_status,runner_rows,provisional_races,first_date,last_date,runner_row_coverage_pct
0,alternative_1_23,16,2,2018-01-06,2018-01-06,0.3467
1,dominant_1_35,4598,587,2018-01-07,2018-12-29,99.6316
2,unresolved,1,1,2018-05-12,2018-05-12,0.0217


## Reverse-engineering feasibility

External validation confirms that foreign prize schedules were converted into stored sterling amounts using a fixed multiplier applied consistently across every paid runner in a race.

For the 2018 United States population:

- 4,598 prize rows across 587 provisional races reconstruct to whole-dollar awards under a `1.35` multiplier;
- 16 prize rows across two Santa Anita races on 6 January reconstruct under `1.23`;
- one runner row remains unresolved.

The dominant `1.35` transformation covers 99.6316% of populated 2018 United States prize rows.

This demonstrates that the original local-currency amounts are potentially recoverable where:

1. the original currency can be assigned confidently;
2. one source multiplier applies consistently to a race, meeting, date or identifiable source batch;
3. reconstructed amounts satisfy strong monetary-rounding tests; and
4. exceptions are retained rather than forced.

However, Notebook 13 will not attempt to reconstruct every historical foreign amount. That would require a governed currency reference, inference rules across all jurisdictions and periods, and independent validation. It should be isolated as a later dependency rather than allowed to dominate the prize-field semantic audit.

In [46]:
# Test whether every populated Irish euro string can be parsed safely.
#
# Irish values are the only populated text values in the source. Before
# defining canonical monetary storage, we need to confirm that removing:
# - the leading euro symbol; and
# - optional thousands commas
#
# always leaves a valid decimal number with no unexpected text.
#
# This cell preserves the original raw value, derives a candidate numeric
# amount, and reports any rows that fail exact pattern validation.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    irish_text_prizes = pd.read_sql_query(
        f"""
        SELECT DISTINCT
            CAST(prize AS TEXT) AS raw_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(prize) = 'text'
          AND TRIM(CAST(prize AS TEXT)) <> ''
        ORDER BY raw_prize
        """,
        conn,
    )

# Accepted forms:
# €99
# €997.50
# €1,000
# €98,875
irish_prize_pattern = r"^€(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{1,2})?$"

irish_text_prizes["matches_expected_format"] = (
    irish_text_prizes["raw_prize"]
    .str.fullmatch(irish_prize_pattern)
)

irish_text_prizes["candidate_amount"] = pd.to_numeric(
    irish_text_prizes["raw_prize"]
    .str.removeprefix("€")
    .str.replace(",", "", regex=False),
    errors="coerce",
)

irish_prize_parse_profile = pd.DataFrame(
    {
        "measure": [
            "distinct populated Irish raw values",
            "values matching expected euro format",
            "values failing expected euro format",
            "values failing numeric conversion",
            "minimum candidate amount",
            "maximum candidate amount",
        ],
        "value": [
            len(irish_text_prizes),
            int(irish_text_prizes["matches_expected_format"].sum()),
            int((~irish_text_prizes["matches_expected_format"]).sum()),
            int(irish_text_prizes["candidate_amount"].isna().sum()),
            irish_text_prizes["candidate_amount"].min(),
            irish_text_prizes["candidate_amount"].max(),
        ],
    }
)

irish_prize_parse_profile

,measure,value
0,distinct populated Irish raw values,1624.0
1,values matching expected euro format,1624.0
2,values failing expected euro format,0.0
3,values failing numeric conversion,0.0
4,minimum candidate amount,10.0
5,maximum candidate amount,855000.0


In [47]:
# Check whether Irish prize strings preserve at most two decimal places and can
# be converted exactly into integer euro cents.
#
# Floating-point values are unsuitable as the canonical monetary representation.
# We therefore parse with Decimal, convert each amount to cents, and verify that
# no value requires fractions smaller than one cent.
#
# A clean result would support storing:
# - raw source value;
# - canonical amount in minor units;
# - currency code EUR; and
# - a confirmed direct-parse transformation method.

from decimal import Decimal, InvalidOperation

def parse_irish_prize_to_cents(raw_value):
    """Parse a validated Irish euro string into exact integer cents."""
    numeric_text = (
        raw_value
        .removeprefix("€")
        .replace(",", "")
    )

    try:
        amount = Decimal(numeric_text)
    except InvalidOperation:
        return None

    cents = amount * 100

    if cents != cents.to_integral_value():
        return None

    return int(cents)

irish_text_prizes["candidate_amount_cents"] = (
    irish_text_prizes["raw_prize"]
    .map(parse_irish_prize_to_cents)
)

irish_minor_unit_profile = pd.DataFrame(
    {
        "measure": [
            "distinct parsed values",
            "values failing exact-cent conversion",
            "minimum amount in cents",
            "maximum amount in cents",
        ],
        "value": [
            len(irish_text_prizes),
            int(
                irish_text_prizes[
                    "candidate_amount_cents"
                ].isna().sum()
            ),
            irish_text_prizes["candidate_amount_cents"].min(),
            irish_text_prizes["candidate_amount_cents"].max(),
        ],
    }
)

irish_minor_unit_profile

,measure,value
0,distinct parsed values,1624
1,values failing exact-cent conversion,0
2,minimum amount in cents,1000
3,maximum amount in cents,85500000


In [48]:
# Test whether populated Great Britain prize values can be represented exactly
# as integer pence.
#
# British values are stored as a mixture of SQLite INTEGER and REAL values.
# We convert through each value's textual representation into Decimal, rather
# than multiplying binary floating-point values directly.
#
# A successful result would support canonical storage as:
# - raw source value;
# - exact amount in minor units;
# - currency code GBP; and
# - direct source interpretation with no inferred FX reversal.

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    british_raw_prizes = pd.read_sql_query(
        f"""
        SELECT DISTINCT
            CAST(prize AS TEXT) AS raw_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND TRIM(CAST(prize AS TEXT)) <> ''
          AND (
              course NOT LIKE '%(IRE)'
              AND course NOT LIKE '%(FR)'
              AND course NOT LIKE '%(USA)'
              AND course NOT LIKE '%(AUS)'
              AND course NOT LIKE '%(CAN)'
              AND course NOT LIKE '%(GER)'
              AND course NOT LIKE '%(JPN)'
              AND course NOT LIKE '%(HK)'
              AND course NOT LIKE '%(UAE)'
          )
        """,
        conn,
    )

# Restrict using the already governed race-level jurisdiction assignments,
# rather than relying on course suffixes alone.
british_race_keys = race_prize_profiles.loc[
    race_prize_profiles["candidate_jurisdiction"] == "Great Britain",
    ["date", "course", "off"],
]

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as conn:
    british_runner_prizes = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            CAST(prize AS TEXT) AS raw_prize
        FROM data
        WHERE {DATA_ROW_PREDICATE}
          AND TRIM(CAST(prize AS TEXT)) <> ''
        """,
        conn,
    )

british_runner_prizes = british_runner_prizes.merge(
    british_race_keys,
    on=["date", "course", "off"],
    how="inner",
    validate="many_to_one",
)

british_distinct_prizes = (
    british_runner_prizes[["raw_prize"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

def parse_decimal_to_minor_units(raw_value):
    """Parse a decimal money string into exact integer hundredths."""
    try:
        amount = Decimal(str(raw_value))
    except InvalidOperation:
        return None

    minor_units = amount * 100

    if minor_units != minor_units.to_integral_value():
        return None

    return int(minor_units)

british_distinct_prizes["candidate_amount_pence"] = (
    british_distinct_prizes["raw_prize"]
    .map(parse_decimal_to_minor_units)
)

british_minor_unit_profile = pd.DataFrame(
    {
        "measure": [
            "distinct populated British raw values",
            "values failing exact-pence conversion",
            "minimum amount in pence",
            "maximum amount in pence",
        ],
        "value": [
            len(british_distinct_prizes),
            int(
                british_distinct_prizes[
                    "candidate_amount_pence"
                ].isna().sum()
            ),
            british_distinct_prizes["candidate_amount_pence"].min(),
            british_distinct_prizes["candidate_amount_pence"].max(),
        ],
    }
)

british_minor_unit_profile

,measure,value
0,distinct populated British raw values,17335
1,values failing exact-pence conversion,0
2,minimum amount in pence,90
3,maximum amount in pence,92153750


In [49]:
# Measure the current canonicalisation status of populated prize rows.
#
# At this stage:
# - Great Britain can be parsed directly as GBP;
# - Ireland can be parsed directly as EUR;
# - all other jurisdictions contain numeric source-presented values whose
#   original currencies and source conversion multipliers are not yet governed.
#
# This cell quantifies how much of the populated field is directly usable and
# how much must remain source-specific pending a separate reconstruction task.

canonicalisation_status = (
    prize_coverage_by_jurisdiction[
        [
            "candidate_jurisdiction",
            "nonblank_prize_rows",
        ]
    ]
    .copy()
)

canonicalisation_status["canonical_status"] = np.select(
    [
        canonicalisation_status["candidate_jurisdiction"]
        == "Great Britain",
        canonicalisation_status["candidate_jurisdiction"]
        == "Ireland",
    ],
    [
        "direct_parse_confirmed_GBP",
        "direct_parse_confirmed_EUR",
    ],
    default="source_presented_amount_currency_unresolved",
)

canonicalisation_summary = (
    canonicalisation_status
    .groupby("canonical_status", dropna=False)
    .agg(
        jurisdictions=("candidate_jurisdiction", "nunique"),
        populated_runner_rows=("nonblank_prize_rows", "sum"),
    )
    .reset_index()
)

canonicalisation_summary["populated_row_coverage_pct"] = (
    100
    * canonicalisation_summary["populated_runner_rows"]
    / canonicalisation_summary["populated_runner_rows"].sum()
).round(2)

canonicalisation_summary

,canonical_status,jurisdictions,populated_runner_rows,populated_row_coverage_pct
0,direct_parse_confirmed_EUR,1,168466,16.65
1,direct_parse_confirmed_GBP,1,561852,55.54
2,source_presented_amount_currency_unresolved,34,281252,27.80


In [50]:
# Measure prize availability and payment depth by race type.
#
# The field meaning is established as runner-level prize allocation, but the
# number of runners receiving a value may vary between Flat, Hurdle, Chase and
# other race types.
#
# For each race type, this cell reports:
# - provisional races;
# - runner rows;
# - populated prize rows;
# - races with any populated prize;
# - races where every runner has a populated prize; and
# - the average number of populated prize rows per covered race.
#
# This tests structural variation without assuming that different race types
# use different monetary semantics.

prize_coverage_by_race_type = (
    race_prize_profiles
    .groupby("type", dropna=False)
    .agg(
        provisional_races=("off", "size"),
        runner_rows=("runner_rows", "sum"),
        populated_prize_rows=("nonblank_prize_rows", "sum"),
        races_with_any_prize=(
            "nonblank_prize_rows",
            lambda values: (values > 0).sum(),
        ),
        races_with_all_rows_prized=(
            "nonblank_prize_rows",
            lambda values: (
                values.to_numpy()
                == race_prize_profiles.loc[
                    values.index,
                    "runner_rows",
                ].to_numpy()
            ).sum(),
        ),
    )
    .reset_index()
)

prize_coverage_by_race_type["race_coverage_pct"] = (
    100
    * prize_coverage_by_race_type["races_with_any_prize"]
    / prize_coverage_by_race_type["provisional_races"]
).round(2)

prize_coverage_by_race_type["average_paid_rows_per_covered_race"] = (
    prize_coverage_by_race_type["populated_prize_rows"]
    / prize_coverage_by_race_type["races_with_any_prize"]
).round(2)

prize_coverage_by_race_type.sort_values(
    "provisional_races",
    ascending=False,
)

,type,provisional_races,runner_rows,populated_prize_rows,races_with_any_prize,races_with_all_rows_prized,race_coverage_pct,average_paid_rows_per_covered_race
1,Flat,126391,1268229,695853,126379,19918,99.99,5.51
2,Hurdle,35462,358441,183095,35462,3462,100.00,5.16
0,Chase,22547,179645,111436,22547,3987,100.00,4.94
3,NH Flat,4643,44970,21186,4643,266,100.00,4.56


In [51]:
# Measure how prize availability and payment depth change over time.
#
# For each calendar year, this cell reports:
# - provisional races;
# - populated prize rows;
# - races with any prize value;
# - races where every runner has a prize value; and
# - average paid runner rows per covered race.
#
# This tests whether the field's structural availability changes materially
# across the source period, without yet interpreting changes in prize amounts.

prize_coverage_by_year = (
    race_prize_profiles
    .assign(year=lambda frame: frame["date"].astype(str).str[:4])
    .groupby("year", dropna=False)
    .agg(
        provisional_races=("off", "size"),
        runner_rows=("runner_rows", "sum"),
        populated_prize_rows=("nonblank_prize_rows", "sum"),
        races_with_any_prize=(
            "nonblank_prize_rows",
            lambda values: (values > 0).sum(),
        ),
        races_with_all_rows_prized=(
            "nonblank_prize_rows",
            lambda values: (
                values.to_numpy()
                == race_prize_profiles.loc[
                    values.index,
                    "runner_rows",
                ].to_numpy()
            ).sum(),
        ),
    )
    .reset_index()
)

prize_coverage_by_year["race_coverage_pct"] = (
    100
    * prize_coverage_by_year["races_with_any_prize"]
    / prize_coverage_by_year["provisional_races"]
).round(2)

prize_coverage_by_year["average_paid_rows_per_covered_race"] = (
    prize_coverage_by_year["populated_prize_rows"]
    / prize_coverage_by_year["races_with_any_prize"]
).round(2)

prize_coverage_by_year.sort_values("year")

,year,provisional_races,runner_rows,populated_prize_rows,races_with_any_prize,races_with_all_rows_prized,race_coverage_pct,average_paid_rows_per_covered_race
0,2015,16609,157683,76285,16608,1711,99.99,4.59
1,2016,16235,157938,75995,16234,1490,99.99,4.68
2,2017,17110,167494,81313,17109,1659,99.99,4.75
3,2018,17546,172047,101639,17545,3672,99.99,5.79
4,2019,17345,173039,102025,17344,3461,99.99,5.88
5,2020,11875,121683,68937,11874,1869,99.99,5.81
6,2021,17706,175552,104907,17704,3877,99.99,5.93
7,2022,17350,167479,94985,17349,3028,99.99,5.47
8,2023,16119,156075,83909,16118,1850,99.99,5.21
9,2024,17170,169702,91994,17169,2012,99.99,5.36


In [52]:
# Separate the yearly payment-depth trend by broad jurisdiction group.
#
# The overall average rises sharply in 2018, but that may be caused by changes
# in the mixture of countries represented rather than a change in the meaning
# or extraction of the `prize` field.
#
# We compare:
# - Great Britain;
# - Ireland; and
# - all other jurisdictions, whose source-presented currency remains unresolved.
#
# For each year and group, the result reports covered races and the average
# number of populated prize rows per covered race.

yearly_jurisdiction_payment_depth = (
    race_prize_profiles
    .assign(
        year=lambda frame: frame["date"].astype(str).str[:4],
        jurisdiction_group=lambda frame: np.select(
            [
                frame["candidate_jurisdiction"] == "Great Britain",
                frame["candidate_jurisdiction"] == "Ireland",
            ],
            [
                "Great Britain",
                "Ireland",
            ],
            default="Other jurisdictions",
        ),
    )
    .loc[lambda frame: frame["nonblank_prize_rows"] > 0]
    .groupby(
        ["year", "jurisdiction_group"],
        dropna=False,
    )
    .agg(
        covered_races=("off", "size"),
        populated_prize_rows=("nonblank_prize_rows", "sum"),
    )
    .reset_index()
)

yearly_jurisdiction_payment_depth[
    "average_paid_rows_per_covered_race"
] = (
    yearly_jurisdiction_payment_depth["populated_prize_rows"]
    / yearly_jurisdiction_payment_depth["covered_races"]
).round(2)

yearly_jurisdiction_payment_depth.pivot(
    index="year",
    columns="jurisdiction_group",
    values="average_paid_rows_per_covered_race",
)

jurisdiction_group,Great Britain,Ireland,Other jurisdictions
year,,,
2015,4.16,4.07,6.00
2016,4.21,4.95,5.79
2017,4.24,5.05,5.81
2018,5.92,5.06,5.94
2019,5.94,5.50,5.98
2020,5.64,5.89,6.06
2021,5.94,5.89,5.92
2022,5.17,5.87,5.92
2023,4.77,5.86,5.99


In [53]:
# Examine the Great Britain change between 2017 and 2018 in more detail.
#
# The yearly averages show a sharp rise in the number of runners with recorded
# prize money per race. This cell counts how many British races had exactly
# 1, 2, 3, 4, 5, 6, or more runners with a recorded prize in each year.
#
# A change concentrated around a specific number of paid runners may reveal
# whether the source began recording additional lower placings from 2018.

british_paid_runner_distribution = (
    race_prize_profiles.loc[
        race_prize_profiles["candidate_jurisdiction"] == "Great Britain"
    ]
    .assign(
        year=lambda frame: frame["date"].astype(str).str[:4],
        recorded_prize_runner_count=lambda frame: (
            frame["nonblank_prize_rows"]
            .clip(upper=7)
            .replace({7: "7+"})
        ),
    )
    .groupby(
        ["year", "recorded_prize_runner_count"],
        dropna=False,
    )
    .size()
    .rename("provisional_races")
    .reset_index()
)

british_paid_runner_distribution.pivot(
    index="year",
    columns="recorded_prize_runner_count",
    values="provisional_races",
).fillna(0).astype(int)

recorded_prize_runner_count,0,1,2,3,4,5,6,7+
year,,,,,,,,
2015,1,4,48,536,8192,413,804,43
2016,1,8,62,423,7995,525,970,51
2017,1,9,82,320,8113,647,1056,60
2018,1,12,89,280,3291,869,1578,4286
2019,1,7,76,280,3194,836,1472,4219
2020,1,3,42,140,2746,386,740,2229
2021,1,7,80,293,3243,880,1567,4282
2022,1,9,99,411,3197,942,5176,319
2023,1,11,73,272,2936,5603,978,141


In [54]:
# Pinpoint when the British recording pattern changed during 2018.
#
# The yearly table shows a sharp move from four recorded prize-money runners
# per race to seven or more. This cell breaks 2017 and 2018 down by month to
# determine whether the change happened abruptly at the start of 2018 or
# partway through the year.
#
# Counts of seven or more runners remain grouped together for readability.

british_monthly_prize_count_distribution = (
    race_prize_profiles.loc[
        (race_prize_profiles["candidate_jurisdiction"] == "Great Britain")
        & (
            race_prize_profiles["date"]
            .astype(str)
            .str[:4]
            .isin(["2017", "2018"])
        )
    ]
    .assign(
        month=lambda frame: frame["date"].astype(str).str[:7],
        recorded_prize_runner_count=lambda frame: (
            frame["nonblank_prize_rows"]
            .clip(upper=7)
            .replace({7: "7+"})
        ),
    )
    .groupby(
        ["month", "recorded_prize_runner_count"],
        dropna=False,
    )
    .size()
    .rename("provisional_races")
    .reset_index()
)

british_monthly_prize_count_distribution.pivot(
    index="month",
    columns="recorded_prize_runner_count",
    values="provisional_races",
).fillna(0).astype(int)

recorded_prize_runner_count,0,1,2,3,4,5,6,7+
month,,,,,,,,
2017-01,0,0,3,22,568,45,42,1
2017-02,0,0,11,35,502,53,45,1
2017-03,0,2,15,41,607,39,59,5
2017-04,0,2,11,41,663,70,96,23
2017-05,0,1,7,36,855,62,116,2
2017-06,0,0,3,26,841,43,109,1
2017-07,0,0,2,10,835,61,117,3
2017-08,0,1,3,13,819,62,124,8
2017-09,1,1,3,14,613,58,127,2


### Why this line of investigation stops here

The number of runners with recorded prize money changes across years and jurisdictions.

In Great Britain, the dominant pattern changes from:

- four runners before 2018;
- seven or more runners from 2018 to 2021;
- six runners in 2022;
- five runners from 2023 onward.

Explaining these changes would require a separate historical investigation into:

- official prize-allocation rules;
- race conditions;
- changes in source coverage; and
- possible upstream extraction changes.

That question is relevant to the history of prize-money distribution, but it is not required to establish the meaning or safe storage of the `prize` field.

Notebook 13 therefore records the variation without attempting to explain it.

The modelling consequence is that the number of runners with recorded prize money must be treated as observed data, not assumed from race type, year or jurisdiction.

## Recommended treatment of the `prize` field

The source field should be interpreted as **runner-level recorded prize money**, not total race prize money.

Recommended staging fields:

- `prize_raw` — original source value preserved unchanged;
- `prize_amount_minor_units` — exact integer pence or cents where the currency is confirmed;
- `prize_currency` — ISO currency code where confirmed;
- `prize_interpretation_status` — direct parse, source-presented amount, or unresolved;
- `prize_conversion_multiplier` — retained only where a source conversion has been independently reconstructed;
- `prize_conversion_status` — confirmed, candidate, or unresolved.

Current confidence:

- Great Britain: direct parse as GBP, confirmed;
- Ireland: direct parse as EUR, confirmed;
- other jurisdictions: preserve the source-presented amount without assigning a currency until jurisdiction-specific validation is complete.

Blank values should remain blank. They must not be converted to zero, because a blank usually means that no prize amount was recorded for that runner, not that the runner received nothing.

The number of runners with recorded prize money must also remain observed data. It varies by race, year and jurisdiction and must not be inferred from finishing position alone.

## Race-level aggregation

Runner prize amounts may be summed within a race to calculate the **total recorded prize money distributed to runners**.

That derived total must not automatically be labelled:

- total race prize money;
- advertised prize fund;
- guaranteed purse; or
- total race value.

Those concepts may include money not represented in the runner rows, while some jurisdictions may record minimum payments to runners finishing outside the principal prize positions.

A race-level derived field should therefore use a precise name such as:

`recorded_runner_prize_total_minor_units`

It should be calculated only where all included runner amounts share a confirmed currency. Races with unresolved source-presented foreign amounts should not be combined with confirmed GBP or EUR totals.

## Findings summary

The `prize` field is usable, but only after separating three distinct cases.

### Confirmed direct values

- Great Britain values are runner-level prize amounts in GBP.
- Ireland values are runner-level prize amounts in EUR.
- Both can be converted exactly into integer minor units.
- No populated British or Irish value requires precision smaller than one penny or cent.

### Foreign source-presented values

Values from other jurisdictions are not safe to interpret directly as local currency.

External checks show that at least some foreign prize schedules were converted before storage. For example, selected United States and French races reconstruct to official local-currency prize schedules only after applying a source multiplier.

These multipliers appear to describe the source transformation, not necessarily the historical market exchange rate.

Foreign values must therefore remain preserved as source-presented amounts until the relevant jurisdiction and period have been validated.

### Missing values

Blank runner values are common and usually indicate that no prize amount was recorded for that runner.

They must remain null and must not be replaced with zero.

### Structural variation

The number of runners with recorded prize money varies by race, year and jurisdiction.

It must be treated as observed source data rather than inferred from finishing position, race type or a fixed number of paid places.

### Overall recommendation

The field should be retained with:

- the unchanged raw source value;
- a confirmed currency only where supported;
- an exact minor-unit amount only where safe;
- an explicit interpretation status; and
- preserved evidence for any later foreign-currency reconstruction.

## Follow-up work

Notebook 13 establishes the field meaning and the safe treatment of currently validated values.

The following questions should be handled separately:

1. **Jurisdiction-specific currency validation**  
   Confirm the official currency and source treatment for each foreign jurisdiction.

2. **Source conversion reconstruction**  
   Determine whether foreign values can be reversed reliably by jurisdiction, date, meeting or source batch.

3. **Historical prize-allocation changes**  
   Investigate why the number of runners with recorded prize money changes over time, especially in Great Britain.

4. **Race-level completeness**  
   Compare summed runner prizes with independently published race purses to determine when the source captures the full distributed total.

These investigations should not alter the raw source values. Any reconstructed amount must remain linked to its method, evidence and confidence level.

In [55]:
# Create a compact decision table for the final notebook output.
#
# This turns the narrative findings into explicit modelling rules that can be
# reused when the staging schema is designed.
#
# The table deliberately separates:
# - what the source field means;
# - what can be parsed safely now;
# - what must remain unresolved; and
# - what must never be inferred automatically.

prize_field_decisions = pd.DataFrame(
    [
        {
            "area": "Field meaning",
            "decision": "Interpret as runner-level recorded prize money",
            "status": "Confirmed",
            "reason": (
                "Values vary between runners within the same race and usually "
                "follow finishing position or jurisdiction-specific payments."
            ),
        },
        {
            "area": "Great Britain",
            "decision": "Parse directly into integer pence with currency GBP",
            "status": "Confirmed",
            "reason": (
                "All 17,335 distinct populated British values convert exactly "
                "to integer pence."
            ),
        },
        {
            "area": "Ireland",
            "decision": "Parse euro-prefixed values into integer cents with currency EUR",
            "status": "Confirmed",
            "reason": (
                "All 1,624 distinct populated Irish values match the expected "
                "format and convert exactly to integer cents."
            ),
        },
        {
            "area": "Other jurisdictions",
            "decision": "Preserve as source-presented amount without assigning currency",
            "status": "Required",
            "reason": (
                "At least some foreign values were converted before storage, "
                "and the source multiplier is not yet governed."
            ),
        },
        {
            "area": "Blank values",
            "decision": "Retain as null rather than replacing with zero",
            "status": "Required",
            "reason": (
                "Blank usually means no amount was recorded for that runner, "
                "not necessarily that the runner received nothing."
            ),
        },
        {
            "area": "Race totals",
            "decision": "Label sums as recorded runner prize totals",
            "status": "Required",
            "reason": (
                "Summed runner values are not automatically equivalent to the "
                "advertised purse or total race value."
            ),
        },
        {
            "area": "Number of paid runners",
            "decision": "Use observed populated runner rows only",
            "status": "Required",
            "reason": (
                "The number of runners with recorded prize money changes by "
                "race, year and jurisdiction."
            ),
        },
        {
            "area": "Foreign reconstruction",
            "decision": "Handle through later jurisdiction-specific studies",
            "status": "Deferred",
            "reason": (
                "Reliable reconstruction requires currency, period, multiplier "
                "and external prize-schedule validation."
            ),
        },
    ]
)

prize_field_decisions

,area,decision,status,reason
0,Field meaning,Interpret as runner-level recorded prize money,Confirmed,Values vary between runners within the same ra...
1,Great Britain,Parse directly into integer pence with currenc...,Confirmed,"All 17,335 distinct populated British values c..."
2,Ireland,Parse euro-prefixed values into integer cents ...,Confirmed,"All 1,624 distinct populated Irish values matc..."
3,Other jurisdictions,Preserve as source-presented amount without as...,Required,At least some foreign values were converted be...
4,Blank values,Retain as null rather than replacing with zero,Required,Blank usually means no amount was recorded for...
5,Race totals,Label sums as recorded runner prize totals,Required,Summed runner values are not automatically equ...
6,Number of paid runners,Use observed populated runner rows only,Required,The number of runners with recorded prize mone...
7,Foreign reconstruction,Handle through later jurisdiction-specific stu...,Deferred,"Reliable reconstruction requires currency, per..."


## Notebook conclusion

Notebook 13 is complete.

It establishes that:

- `prize` records runner-level prize money;
- Great Britain values can be parsed directly as GBP;
- Ireland values can be parsed directly as EUR;
- British and Irish values can be stored exactly as integer minor units;
- foreign values must remain source-presented until validated by jurisdiction and period;
- blank values must remain null;
- race-level sums must be labelled as recorded runner prize totals;
- and the number of runners with recorded prize money must not be assumed.

The notebook also identifies three separate follow-up investigations:

1. jurisdiction-specific foreign-currency reconstruction;
2. historical changes in the number of runners receiving recorded prize money;
3. comparison of summed runner prizes with independently published race purses.

These are intentionally deferred so that the core field audit remains narrow, reproducible and usable for staging-schema design.